# Shadow Estimation with the Platonic-Solid POVMs — Walkthrough

A companion walkthrough for `code/shadow_experiments.py`, the numerical study behind Section 5.2
and Appendix F.3 of the thesis (only the two-protocol study is in print; the rest is
repository-only). A reader who works through this notebook instead of the script misses out on
**nothing**: every function of the module appears below verbatim (the builder lifts them
mechanically from the source at build time), every experiment is re-run in full, and the final
cell proves the point by rebuilding the study's entire output dictionary and diffing it against
the committed `data/shadow_experiments.npz` — key for key, value for value, to exactly zero. The
notebook writes nothing; the script owns the npz.

The whole appendix hangs off one skeleton:

> **1 lemma, 2 conjugations, 3 corners, 5 studies.**

One corollary of Schur's lemma. Two things a random rotation can wrap into a group average — the
readout *composed with* the noise (randomized-projective, scalar $T_{zz}$), or the noise *alone*
(twirled-native, scalar $\operatorname{tr}T/3$). Three implementation corners (native /
twirled-native / randomized-projective). Five numbered studies, one job each: the variance
landscape, the dual-frame optimization, the robust calibration, its blind spot, and the scaling
check. Hold the skeleton and everything else here is a corollary;
hold only the numbers and you hold nothing.

**References:**
- Huang, Kueng & Preskill, *Predicting many properties of a quantum system from very few measurements* (2020) — classical shadows, the $3^w$ sampling cost
- Nguyen, Bönsel, Steinberg & Gühne, *Optimising shadow tomography with generalised measurements* (2022) — Platonic-solid POVM shadows, the closest adjacent work
- Chen, Yu, Zeng & Flammia, *Robust shadow estimation* (2021) — calibration under a measurement twirl; their random-Clifford primitive is our octahedral protocol
- Innocenti et al., *Shadow tomography on general measurement frames* (2023) — the closed-form variance for 3-design measurements this study's Experiment 1 extends
- D'Ariano, Perinotti & Sacchi (2005) — the tetrahedral SIC as an *indecomposable* POVM
- Korhonen et al. (2025) — locally-optimal duals losing on composite observables (our 1.069 cell, arrived at uninvited)
- Caprotti et al. (2026); Mangini et al. (2025) — dual-optimization bounds, and what leaving the factorized class buys
- Wilkens et al. (2026) — local robust shadows run on trapped-ion hardware
- Brieger et al. (2025) — silent failure of miscalibrated robust estimators
- Jeanette et al. (2026) — blind calibration, the other exit from the blind spot
- Decker, Janzing & Beth (2004) — the native (Naimark) measurement circuits
- Elben et al., *The randomized measurement toolbox* (2022) — randomized-projective's literature home

## 0. One reframing, and the map

**A POVM is its Bloch-vertex array `s`** of shape $(V, 3)$. Everything in the study is a small
linear-algebra statement about `s`:

- *effects*: $E_k = \frac1V(\mathrm{Id} + \hat n_k\cdot\vec\sigma)$ — the array row-for-row;
- *probabilities*: $p(k) = \frac1V(1 + \hat n_k\cdot r)$ — affine in the state's Bloch vector;
- *dual frames*: pairs $(\alpha_k, \beta_k)$ with $D_k = \alpha_k\,\mathrm{Id} + \beta_k\cdot\vec\sigma$
  — unbiasedness is 16 linear equations in the $4V$ coordinates, variance a quadratic form, so
  optimizing the dual is a linearly-constrained quadratic program;
- *noise*: an affine Bloch map $r \to Tr + t$, pushed onto the effects in the Heisenberg picture;
- *$n$-qubit Born distributions*: one `einsum` of $\rho$ against per-wire effect tensors — so
  single-shot means and second moments are **finite sums evaluated exactly**, and every
  deterministic claim (unbiasedness, variance identities, bias factors, twirl scalars) is a
  theorem checked to machine precision, with Monte Carlo only measuring convergence *to targets
  already known*;
- *gate circuits*: affine Bloch maps too — so each randomized implementation's effective channel is
  an exact finite group average, no sampling anywhere.

| § | study | its one job |
|---|---|---|
| 3 | Experiment 1 + the exact landscape | does the choice of solid matter at the canonical dual? (provably: no) |
| 4 | the fourth-moment ladder | where the solid choice *does* reappear: the tails |
| 5 | Experiment 2 + exact ratios | what the $(4V{-}16)$-dimensional dual family buys (little, and not for the reason you'd guess) |
| 6 | Experiment 3 | robust calibration: the exact identity, and what the insurance premium costs |
| 7 | the blind spot + free diagnostic | where a $\ket{0}$-calibration is structurally blind, and the diagnostic hiding in its discards |
| 8 | the twirl + the two protocols | one lemma, two conjugations: $T_{zz}$ versus $\operatorname{tr}T/3$ |
| 9 | gate noise | the twirl's one assumption, priced with the atlas — and *why* the twirled-native residual is second order |
| 10 | $n$-scaling | nothing above is an $n=4$ artifact |
| 11 | the receipts | this notebook = the committed npz, exactly |

**The random-number discipline.** One shared `rng` is created in §2 and consumed *strictly in the
production order* of the script's `main()` — `make_states` → Experiment 1 → Experiment 2 →
Experiment 3 → blindness — and never touched anywhere else; every demonstration cell uses its own
local generator. That discipline is what makes §11's value-for-value replay possible. Everything
from §8 onward is deterministic (exact group averages; the two local generators inside
`twirl_check` and `two_protocol_twirl` are fixed-seed irreducibility witnesses, not Monte Carlo).

In [1]:
# === Setup (lifted from shadow_experiments.py): imports + constants ===

import numpy as np
from pathlib import Path
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import eigsh

DATA = Path("data")   # the one adaptation: the module resolves this next to itself

RNG_SEED = 20260612

N_QUBITS = 4

R_REPS = 2000         # Monte Carlo replications per (POVM, state, setting)

T_MAIN = 1000         # shots per replication (Experiments 1 and 3)

T_EXP2 = 5000         # shots per replication (Experiment 2 headline)

RC_CAL = 1000         # calibration shots per replication (Experiment 3)

N_HAAR = 10           # Haar-random states in Experiment 1

G_CRIT = 1.0          # TFIM transverse field (critical point)

TRAIN_G = (0.3, 0.5, 0.7, 1.0, 1.5)   # training TFIM fields (+ GHZ)

DEPOL_RATES = (0.0, 0.01, 0.05, 0.1, 0.2)

BLIND_RATES = (0.01, 0.05, 0.1, 0.2)

GATE_GAMMAS = (0.0, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05)

GAMMA_DIL = 0.05      # twirled-native series: modeled dilation pre-channel

INDEP_DELTAS = (0.01, 0.05, 0.2)   # gate-INDEPENDENT control: swept, not pinned

NSCALE_NS = (4, 6, 8, 10, 12, 14, 16)

SOLIDS = ("tetrahedron", "octahedron", "cube", "icosahedron", "dodecahedron")

ANTIPODAL = SOLIDS[1:]          # vertex sets closed under n -> -n

I2 = np.eye(2, dtype=complex)

PAULI = {
    "I": I2,
    "X": np.array([[0, 1], [1, 0]], dtype=complex),
    "Y": np.array([[0, -1j], [1j, 0]], dtype=complex),
    "Z": np.array([[1, 0], [0, -1]], dtype=complex),
}

AXIS = {"X": 0, "Y": 1, "Z": 2}

## 1. The five solids, as arrays

The vertices and effects come from the thesis's own symbolic exports (`data/povm_*.npz`, written
by `export_numpy.py` from `povm_properties.py`'s symbolic vertex orderings, in the atlas
orientation — the one where $\Phi$'s rotation axis lands *on* an icosahedron vertex). The loader
pins down the four facts that carry the whole study: every vertex is a unit vector, the vertex
sum is **zero** (the solid is centered), the vertex covariance is **isotropic**,
$\sum_k \hat n_k \hat n_k^\top = \frac{V}{3}\,\mathrm{Id}$ — the 2-design property — and effect
$E_k$ is built from vertex $\hat n_k$ at the *same index* $k$, which is the join every estimator
below relies on and the only one of the four a permutation of the rows can break. Completeness
of the effects is checked alongside.

In [2]:
# === POVM data (Bloch vertices + effects from the symbolic exports) ===

def load_povms():
    """Load Bloch vertices + effects from the symbolic-derived exports."""
    povms = {}
    for name in SOLIDS:
        d = np.load(DATA / f"povm_{name}.npz")
        s, E = d["vertices"], d["elements"]
        V = len(s)
        # Deviations are measured and bounded, not handed to allclose, which
        # keeps rtol=1e-5 alongside any atol and scales it with the expected
        # operand.  Worst over the five solids: 1.1e-16 (norm), 1.8e-15
        # (2-design), 2.2e-16 (completeness).  The zero-sum check keeps
        # allclose: its reference is 0.0, so atol really is the bound.
        d_norm = np.abs(np.linalg.norm(s, axis=1) - 1.0).max()
        assert d_norm < 1e-12, f"{name}: vertices not unit norm " \
            f"(max |dev| = {d_norm:.2e})"
        assert np.allclose(s.sum(axis=0), 0.0, atol=1e-12), name
        d_design = np.abs(s.T @ s - (V / 3) * np.eye(3)).max()      # 2-design
        assert d_design < 1e-12, f"{name}: not a spherical 2-design " \
            f"(max |dev| = {d_design:.2e})"
        d_complete = np.abs(E.sum(axis=0) - I2).max()            # completeness
        assert d_complete < 1e-12, f"{name}: POVM not complete " \
            f"(max |dev| = {d_complete:.2e})"
        # E[k] is built from s[k], at the same index.  The three checks above
        # are permutation-invariant sums; this is the only one row order breaks.
        E_from_s = (I2[None] + np.einsum(
            "kn,nab->kab", s, np.stack([PAULI["X"], PAULI["Y"], PAULI["Z"]]))) / V
        d_align = np.abs(E - E_from_s).max()
        assert d_align < 1e-12, f"{name}: effects not row-for-row aligned with " \
            f"vertices (max |dev| = {d_align:.2e})"
        povms[name] = {"V": V, "s": s, "E": E}
    print(f"  POVMs loaded: " + ", ".join(
        f"{n} (V={p['V']})" for n, p in povms.items()))
    return povms


povms = load_povms()

  POVMs loaded: tetrahedron (V=4), octahedron (V=6), cube (V=8), icosahedron (V=12), dodecahedron (V=20)


In [3]:
# === Demo: the whole estimation story in three matvecs (local rng only) ===

rng_demo = np.random.default_rng(6)
s = povms["icosahedron"]["s"]
print("centered:  max|sum of vertices| =", np.abs(s.sum(axis=0)).max())
print("2-design:  max|s^T s - (V/3) Id| =",
      np.abs(s.T @ s - (len(s) / 3) * np.eye(3)).max())

r = rng_demo.standard_normal(3)
r *= rng_demo.random() / np.linalg.norm(r)      # a random state in the ball
p = (1 + s @ r) / len(s)                        # Born probabilities: affine in r
assert abs(p.sum() - 1) < 1e-12 and p.min() > 0
print("\nE[sampled vertex] =", np.round(p @ s, 8))
print("            r / 3 =", np.round(r / 3, 8))
print("\nthe 1/3 shrinkage is the 2-design identity at work; tripling the "
      "sampled vertex\nundoes it -- that rescaling IS the canonical dual, "
      "and it is unbiased by geometry")

centered:  max|sum of vertices| = 0.0
2-design:  max|s^T s - (V/3) Id| = 1.3322676295501878e-15

E[sampled vertex] = [ 0.04003202  0.06752964 -0.09705811]
            r / 3 = [ 0.04003202  0.06752964 -0.09705811]

the 1/3 shrinkage is the 2-design identity at work; tripling the sampled vertex
undoes it -- that rescaling IS the canonical dual, and it is unbiased by geometry


## 2. States, observables, and the shared stream

Four qubits throughout. The observables: single Paulis $Z_0, X_0$, the strings $Z_0Z_1$,
$X_0X_1$, $Z_0Z_1Z_2$, and the critical TFIM energy
$H = -\sum_i Z_iZ_{i+1} - h\sum_i X_i$ at $h = 1$. The states: the TFIM ground state, GHZ, a
generic product state, and ten Haar-random states shared across all five POVMs. Everything any
estimator reports is checked against exact values computed from the dense operators.

The next cell defines the machinery; the one after it starts **the shared Monte Carlo stream**
(seed `20260612`). From here on, that stream is consumed only by the five marked production
cells, in order.

In [4]:
# === Observables and states ===

def kron_all(ops):
    out = ops[0]
    for o in ops[1:]:
        out = np.kron(out, o)
    return out


def term_operator(sites):
    """Dense operator for a Pauli term given as [(site, letter), ...]."""
    letters = ["I"] * N_QUBITS
    for i, a in sites:
        letters[i] = a
    return kron_all([PAULI[a] for a in letters])


def obs_operator(obs):
    return sum(c * term_operator(t) for c, t in obs)


def exact_value(rho, obs):
    val = np.trace(rho @ obs_operator(obs))
    assert abs(val.imag) < 1e-12
    return val.real


def tfim_terms(g):
    """4-qubit periodic TFIM:  H = -sum ZZ - g sum X."""
    terms = [(-1.0, [(i, "Z"), ((i + 1) % N_QUBITS, "Z")])
             for i in range(N_QUBITS)]
    terms += [(-g, [(i, "X")]) for i in range(N_QUBITS)]
    return terms


OBSERVABLES = {
    "Z0":     [(1.0, [(0, "Z")])],
    "X0":     [(1.0, [(0, "X")])],
    "Z0Z1":   [(1.0, [(0, "Z"), (1, "Z")])],
    "X0X1":   [(1.0, [(0, "X"), (1, "X")])],
    "Z0Z1Z2": [(1.0, [(0, "Z"), (1, "Z"), (2, "Z")])],
    "E_TFIM": tfim_terms(G_CRIT),
}


def tfim_ground_state(g):
    H = obs_operator(tfim_terms(g))
    _, vecs = np.linalg.eigh(H)
    return vecs[:, 0]


def density(psi):
    return np.outer(psi, psi.conj())


def make_states(rng):
    ghz = np.zeros(16, dtype=complex)
    ghz[0] = ghz[15] = 1 / np.sqrt(2)
    plus = np.array([1, 1]) / np.sqrt(2)
    plus_i = np.array([1, 1j]) / np.sqrt(2)
    zero, one = np.array([1, 0]), np.array([0, 1])
    product = kron_all([plus, zero, plus_i, one]).astype(complex)
    states = {
        "TFIM": density(tfim_ground_state(G_CRIT)),
        "GHZ": density(ghz),
        "product": density(product),
    }
    haar = []
    for _ in range(N_HAAR):
        psi = rng.standard_normal(16) + 1j * rng.standard_normal(16)
        haar.append(density(psi / np.linalg.norm(psi)))
    return states, haar


def site_bloch(rho):
    """Per-site Bloch vectors (N_QUBITS, 3) of the reduced states."""
    r = np.empty((N_QUBITS, 3))
    for i in range(N_QUBITS):
        for a in "XYZ":
            r[i, AXIS[a]] = exact_value(rho, [(1.0, [(i, a)])])
    return r

In [5]:
# === THE SHARED STREAM -- created once, consumed strictly in production order ===
# (make_states -> exp1 -> exp2 -> exp3 -> blindness; demos use local rngs.)

rng = np.random.default_rng(RNG_SEED)
states, haar = make_states(rng)                                # [stream 1/5]

print(f"TFIM h=1 ground energy: "
      f"{exact_value(states['TFIM'], OBSERVABLES['E_TFIM']):.6f}")
print(f"TFIM site-0 Bloch vector r0 = "
      f"{np.round(site_bloch(states['TFIM'])[0], 5)}   "
      "(the gate-noise study's test vector)")

TFIM h=1 ground energy: -5.226252
TFIM site-0 Bloch vector r0 = [0.65328 0.      0.     ]   (the gate-noise study's test vector)


## 2a. Noise, Born tensors, duals, estimators

**Noise** is an affine Bloch map $r \to Tr + t$ applied i.i.d. per qubit just before the
measurement. Four channels appear: depolarizing ($T = (1-p)\,\mathrm{Id}$), dephasing
($T = \mathrm{diag}(1{-}2p, 1{-}2p, 1)$), amplitude damping
($T = \mathrm{diag}(\sqrt{1{-}\gamma}, \sqrt{1{-}\gamma}, 1{-}\gamma)$, $t = \gamma\hat z$ — the
only one with a shift), and a *tilted* depolarizing channel for the diagnostic of §7. In the
Heisenberg picture the noise moves onto the effects:
$\tilde E_k = \frac1V[(1 + \hat n_k\cdot t)\,\mathrm{Id} + (T^\top \hat n_k)\cdot\vec\sigma]$.

**Born distributions.** The $n$-qubit outcome distribution of the product POVM is an explicit
$(V,V,V,V)$ tensor — one `einsum`. That tensor is the study's method in miniature: any
*single-shot* moment of any estimator is a finite sum over it, evaluated term by term. Sampling
(`sample_outcomes`) exists only to measure convergence speed.

In [6]:
# === Noise channels: affine Bloch maps, applied to the effects ===

def chan_depolarizing(p):
    return (1 - p) * np.eye(3), np.zeros(3)


def chan_dephasing(p):
    """rho -> (1-p) rho + p Z rho Z."""
    return np.diag([1 - 2 * p, 1 - 2 * p, 1.0]), np.zeros(3)


def chan_amp_damping(g):
    return np.diag([np.sqrt(1 - g), np.sqrt(1 - g), 1 - g]), \
        np.array([0.0, 0.0, g])


def chan_tilted_depolarizing(p, theta):
    """Depolarizing composed with a rotation about x: tilts the z-axis."""
    c, s = np.cos(theta), np.sin(theta)
    R = np.array([[1, 0, 0], [0, c, -s], [0, s, c]])
    return (1 - p) * R, np.zeros(3)


def noisy_effects(s, T, t):
    """Heisenberg-picture effects E~_k = (1/V)[(1 + s_k.t) Id + (T^T s_k).sigma]."""
    V = len(s)
    c = 1.0 + s @ t
    s_eff = s @ T                       # rows are T^T s_k
    E = (c[:, None, None] * I2
         + s_eff[:, 0, None, None] * PAULI["X"]
         + s_eff[:, 1, None, None] * PAULI["Y"]
         + s_eff[:, 2, None, None] * PAULI["Z"]) / V
    return E

In [7]:
# === Born distributions and sampling ===

def born_tensor(rho, E):
    """Joint outcome distribution (V,V,V,V) for i.i.d. per-qubit effects E."""
    p = np.einsum("ijklmnop,ami,bnj,cok,dpl->abcd",
                  rho.reshape([2] * (2 * N_QUBITS)), E, E, E, E)
    assert np.abs(p.imag).max() < 1e-12
    p = p.real
    assert p.min() > -1e-12 and abs(p.sum() - 1) < 1e-10
    return np.clip(p, 0.0, None)


def sample_outcomes(p, n_shots, rng):
    """(n_shots, N_QUBITS) outcome indices drawn from the joint tensor."""
    cum = np.cumsum(p.ravel())
    cum[-1] = 1.0
    idx = np.searchsorted(cum, rng.random(n_shots), side="right")
    return np.stack(np.unravel_index(idx, p.shape), axis=1)


def single_qubit_probs(s, T, t, r0):
    """Outcome distribution of one noisy POVM on a product-state qubit r0."""
    q = (1.0 + s @ (T @ r0 + t)) / len(s)
    assert q.min() > -1e-12 and abs(q.sum() - 1) < 1e-12
    return np.clip(q, 0.0, None)

### The dual family: 16 equations in $4V$ unknowns

A single-qubit dual frame is $D_k = \alpha_k\,\mathrm{Id} + \beta_k\cdot\vec\sigma$: four numbers
per outcome, $4V$ in all. Unbiasedness — the frame condition
$\sum_k \operatorname{tr}[E_k\rho]\,D_k = \rho$ for every $\rho$ — is a $4\times4$ reconstruction
map set entry-by-entry to the identity: **16 linear equations** (`frame_system`). What remains is
an affine family of valid duals of dimension $4V - 16$: a *point* for the tetrahedron
($V = d^2 = 4$, the control), $8/16/32/64$ dimensions for the four overcomplete solids. The
canonical dual $\alpha = \frac12$, $\beta = \frac32\hat n_k$ is one member; `optimize_dual` finds
the variance-minimizing member for one Pauli letter by solving the quadratic program — since the
mean is pinned by the constraints, minimizing the second moment *is* minimizing the variance, and
the QP reduces to a linear solve in the null space of the constraint matrix.

In [8]:
# === Dual frames: canonical, robust, and the variance-minimizing QP ===

def canonical_dual(s):
    return 0.5 * np.ones(len(s)), 1.5 * s


def robust_canonical_dual(s, eta):
    """Inverts effective shrinkage eta instead of 1/3 (eta = 1/3 is noiseless)."""
    return 0.5 * np.ones(len(s)), s / (2 * eta)


def frame_system(s_eff):
    """Constraint system A v = b for duals of effects (1/V)(Id + s_eff.sigma)."""
    V = len(s_eff)
    A = np.zeros((16, 4 * V))
    b = np.zeros(16)
    A[0, :V] = 1.0                       # sum alpha = V/2
    b[0] = V / 2
    for j in range(3):                   # sum alpha_k s_eff_kj = 0
        A[1 + j, :V] = s_eff[:, j]
    for i in range(3):                   # sum_k beta_ki = 0
        A[4 + i, V + i::3] = 1.0
    row = 7
    for i in range(3):                   # sum_k beta_ki s_eff_kj = (V/2) d_ij
        for j in range(3):
            A[row, V + i::3] = s_eff[:, j]
            b[row] = V / 2 if i == j else 0.0
            row += 1
    return A, b


def optimize_dual(s_eff, pbar, axis):
    """Variance-minimizing dual for one Pauli letter.

    Minimizes the second moment sum_k pbar_k tr(sigma_axis D_k)^2 over the
    affine family of frame duals (the mean is fixed by unbiasedness, so this
    is the variance).  Quadratic objective + linear constraints = particular
    solution + null-space linear solve.
    """
    V = len(s_eff)
    A, b = frame_system(s_eff)
    v_p, *_ = np.linalg.lstsq(A, b, rcond=None)
    _, sv, Vt = np.linalg.svd(A)
    rank = int((sv > 1e-10 * sv[0]).sum())
    N = Vt[rank:].T                            # null-space basis (4V, 4V-rank)
    q = np.zeros(4 * V)                        # objective: tr(sigma_a D_k) = 2 beta_ka
    q[V + axis::3] = 4.0 * pbar
    M = N.T @ (q[:, None] * N)
    rhs = -N.T @ (q * v_p)
    c, *_ = np.linalg.lstsq(M, rhs, rcond=None)
    v = v_p + N @ c
    assert np.linalg.norm(A @ v - b) < 1e-9
    return v[:V], v[V:].reshape(V, 3)


def dual_frame_residual(alpha, beta, s):
    """max |sum_k tr(E_k rho) D_k - rho| over random states (unbiasedness)."""
    rng = np.random.default_rng(5)
    V = len(s)
    worst = 0.0
    for _ in range(5):
        r = rng.standard_normal(3)
        r /= np.linalg.norm(r) / rng.random()       # random point in the ball
        rho = (I2 + r[0] * PAULI["X"] + r[1] * PAULI["Y"]
               + r[2] * PAULI["Z"]) / 2
        pk = (1.0 + s @ r) / V
        recon = sum(
            pk[k] * (alpha[k] * I2 + beta[k, 0] * PAULI["X"]
                     + beta[k, 1] * PAULI["Y"] + beta[k, 2] * PAULI["Z"])
            for k in range(V))
        worst = max(worst, np.abs(recon - rho).max())
    return worst

In [9]:
# === Demo: how big is each dual family? (deterministic) ===

for name, povm in povms.items():
    s, V = povm["s"], povm["V"]
    A, b = frame_system(s)
    rank = np.linalg.matrix_rank(A, tol=1e-10)
    alpha, beta = canonical_dual(s)
    v_can = np.concatenate([alpha, beta.ravel()])
    assert np.abs(A @ v_can - b).max() < 1e-12   # the canonical dual qualifies
    print(f"  {name:12s} V = {V:2d}:  4V = {4 * V:3d} coordinates,  "
          f"rank(A) = {rank},  free dimensions = {4 * V - rank}")
print("\nthe tetrahedron's family is a single point (unique dual, the control);"
      "\nExperiment 2 searches the others")

  tetrahedron  V =  4:  4V =  16 coordinates,  rank(A) = 16,  free dimensions = 0
  octahedron   V =  6:  4V =  24 coordinates,  rank(A) = 16,  free dimensions = 8
  cube         V =  8:  4V =  32 coordinates,  rank(A) = 16,  free dimensions = 16
  icosahedron  V = 12:  4V =  48 coordinates,  rank(A) = 16,  free dimensions = 32
  dodecahedron V = 20:  4V =  80 coordinates,  rank(A) = 16,  free dimensions = 64

the tetrahedron's family is a single point (unique dual, the control);
Experiment 2 searches the others


### Estimator conventions

A *dual assignment* is a lookup table `lut[i, a, k]`: the per-shot value site $i$ contributes when
reading Pauli letter $a$ off outcome $k$. With the canonical dual that value is
$3(\hat n_k)_a$; identity sites contribute exactly $1$ (every dual here keeps
$\operatorname{tr}D_k = 1$), so they fall away for free. A composite observable is estimated
term by term, each term multiplying its sites' letter-values — and `exact_estimator_mean` /
`exact_second_moment` evaluate the same contractions *exactly* over the outcome tensor, which is
what turns every unbiasedness and variance claim below into a deterministic assert.

One deliberate convention, against the more obvious alternative: duals are pure classical
post-processing, so each term of a composite observable uses the dual matched to *its own letter at
each site*, rather than routing each qubit to a single dual for all terms (that of its most
frequent letter). Single-term observables coincide either way; the TFIM energy does not.
Unbiasedness is unaffected and the optimized levels can only improve.

In [10]:
# === Dual assignments (LUTs) and estimator evaluation ===

def lut_uniform(x_axes):
    """Same per-letter values on every site; x_axes has shape (3, V)."""
    return np.broadcast_to(x_axes, (N_QUBITS,) + x_axes.shape)


def lut_canonical(s):
    return lut_uniform(3.0 * s.T)


def lut_robust_canonical(s, eta):
    return lut_uniform(s.T / eta)


def lut_from_letter_duals(betas):
    """betas: dict axis -> (V, 3) optimized beta for that letter."""
    V = next(iter(betas.values())).shape[0]
    x = np.empty((3, V))
    for a in range(3):
        x[a] = 2.0 * betas[a][:, a]
    return lut_uniform(x)


def shot_estimates(outcomes, obs, lut):
    total = np.zeros(len(outcomes))
    for coeff, sites in obs:
        v = np.full(len(outcomes), coeff)
        for i, a in sites:
            v *= lut[i, AXIS[a]][outcomes[:, i]]
        total += v
    return total


def exact_estimator_mean(p, obs, lut):
    """E[single-shot estimate], summed exactly over the outcome tensor."""
    V = p.shape[0]
    total = 0.0
    for coeff, sites in obs:
        f = [np.ones(V)] * N_QUBITS
        for i, a in sites:
            f = list(f)
            f[i] = lut[i, AXIS[a]]
        total += coeff * np.einsum("abcd,a,b,c,d", p, *f)
    return total


def exact_second_moment(p, obs, lut):
    """E[(single-shot estimate)^2], summed exactly over the outcome tensor.

    Same contraction as exact_estimator_mean, applied to every ordered pair
    of terms: a site hit by both terms multiplies its two letter-values.
    Together with the exact mean this makes single-shot variances -- and so
    every variance identity in the appendix -- deterministic statements.
    """
    V = p.shape[0]
    total = 0.0
    for c1, sites1 in obs:
        for c2, sites2 in obs:
            f = [np.ones(V) for _ in range(N_QUBITS)]
            for i, a in sites1:
                f[i] = f[i] * lut[i, AXIS[a]]
            for i, a in sites2:
                f[i] = f[i] * lut[i, AXIS[a]]
            total += c1 * c2 * np.einsum("abcd,a,b,c,d", p, *f)
    return total


def rep_stats(flat, truth):
    """bias^2 / variance / MSE of the T-shot mean across R replications."""
    means = flat.reshape(R_REPS, -1).mean(axis=1)
    err2 = (means - truth) ** 2
    return {
        "bias2": (means.mean() - truth) ** 2,
        "var": means.var(ddof=1),
        "mse": err2.mean(),
        "se_mse": err2.std(ddof=1) / np.sqrt(R_REPS),
    }

## 3. Experiment 1 — the canonical dual is blind to the choice of solid

**The job:** measure fixed states with all five solids at the canonical dual — no noise, no
optimization — and ask whether the choice of solid matters. The table invites a ranking; the
theorem says there is none to be had.

**The theorem (the site-cases identity).** Expand the estimator's second moment over ordered pairs
of Pauli strings; each pair contributes $\operatorname{tr}[\rho\bigotimes_i M_i]$ with one
single-qubit operator per site, fixed by which strings touch it:

$$M_i = \begin{cases}
\mathrm{Id} & \text{neither string touches site } i,\\[2pt]
\sum_k 3(\hat n_k)_a E_k = \sigma_a & \text{one does, with letter } a,\\[2pt]
\sum_k 9(\hat n_k)_a(\hat n_k)_b E_k = 3\delta_{ab}\,\mathrm{Id} + 9\,S_{abc}\,\sigma_c
  & \text{both do, with letters } a, b,
\end{cases}$$

where $S_{abc} = \frac1V\sum_k (\hat n_k)_a(\hat n_k)_b(\hat n_k)_c$ is the vertex set's
third-moment tensor. The middle case is the frame condition read backwards — it holds for *every*
valid dual, which §10 will lean on. So the vertices enter the variance through their moments of
order **at most three**: order 1 vanishes (centered), order 2 is universal (2-design), and $S$
vanishes for every antipodal vertex set. Consequences, all asserted exactly:

- octahedron, cube, icosahedron, dodecahedron: **identical variance on every (state, observable)**;
- a lone weight-$w$ Pauli string sits at $\operatorname{Var} = 3^w - \langle P\rangle^2$ exactly,
  for *all five* solids (the tetrahedron's $(\hat n)_a^2 = \frac13$ kills its same-letter third
  moments) — the familiar $3^w$ Pauli-shadow cost, here an identity rather than a bound, owned by
  the whole family;
- the tetrahedron can deviate **only** where all three conditions meet: a term pair sharing a site
  with two *different* letters (among our observables: only the TFIM energy, where $ZZ$ meets
  $X$), a state whose amplitudes are not all real (the surviving operator carries a lone
  $\sigma_y$ — so not TFIM or GHZ, only the Haar draws), and a nonvanishing $S$ (only the
  tetrahedron). One cell of the whole table.

**Counterfactual** (worth holding): a vertex set that is a 2-design but *not centered* breaks the
middle case — $\sum_k 3(\hat n_k)_a E_k$ picks up a multiple of the identity, so the canonical
formula is no longer unbiased, and the $\ket{0}$-calibration of §6 reads
$\bar n_z + \frac13 \neq \frac13$ *on a clean channel*. Centering is not decoration; it is load-bearing.

In [11]:
# === Experiment 1: the Monte Carlo landscape ===

def experiment_1(povms, states, haar, rng):
    print("\nExperiment 1: variance landscape, canonical dual "
          f"(shots={T_MAIN}, reps={R_REPS})")
    obs_names = list(OBSERVABLES)
    truths = {(sn, on): exact_value(rho, OBSERVABLES[on])
              for sn, rho in states.items() for on in obs_names}

    mse = {}        # (povm, state, obs) -> stats
    haar_mse = {}   # (povm, obs) -> mean MSE over Haar states
    for pname, povm in povms.items():
        lut = lut_canonical(povm["s"])
        for sname, rho in states.items():
            p = born_tensor(rho, povm["E"])
            for on in obs_names:   # deterministic unbiasedness, no sampling
                em = exact_estimator_mean(p, OBSERVABLES[on], lut)
                assert abs(em - truths[(sname, on)]) < 1e-9, (pname, sname, on)
            out = sample_outcomes(p, R_REPS * T_MAIN, rng)
            for on in obs_names:
                mse[(pname, sname, on)] = rep_stats(
                    shot_estimates(out, OBSERVABLES[on], lut),
                    truths[(sname, on)])
        acc = {on: [] for on in obs_names}
        for rho in haar:
            p = born_tensor(rho, povm["E"])
            out = sample_outcomes(p, R_REPS * T_MAIN, rng)
            for on in obs_names:
                acc[on].append(rep_stats(
                    shot_estimates(out, OBSERVABLES[on], lut),
                    exact_value(rho, OBSERVABLES[on]))["mse"])
        for on in obs_names:
            haar_mse[(pname, on)] = np.mean(acc[on])
    print("  exact unbiasedness of the canonical dual: OK "
          "(all POVMs x states x observables, deterministic)")

    def table(title, getter):
        print(f"\n  MSE, {title}:")
        print("    " + f"{'':10s}" + "".join(f"{n[:12]:>14s}" for n in SOLIDS))
        for on in obs_names:
            print(f"    {on:10s}" + "".join(
                f"{getter(pn, on):14.5f}" for pn in SOLIDS))

    table("TFIM ground state (h=1)", lambda pn, on: mse[(pn, 'TFIM', on)]["mse"])
    table("Haar average", lambda pn, on: haar_mse[(pn, on)])

    print("\n  Haar average, ratio to octahedron / CV across POVMs:")
    for on in obs_names:
        vals = np.array([haar_mse[(pn, on)] for pn in SOLIDS])
        ratios = vals / haar_mse[("octahedron", on)]
        cv = vals.std(ddof=1) / vals.mean()
        print(f"    {on:10s}" + "".join(f"{r:14.3f}" for r in ratios)
              + f"   CV = {cv:5.1%}")
    return mse, haar_mse


exp1, exp1_haar = experiment_1(povms, states, haar, rng)        # [stream 2/5]


Experiment 1: variance landscape, canonical dual (shots=1000, reps=2000)


  exact unbiasedness of the canonical dual: OK (all POVMs x states x observables, deterministic)

  MSE, TFIM ground state (h=1):
                 tetrahedron    octahedron          cube   icosahedron  dodecahedron
    Z0               0.00305       0.00305       0.00298       0.00300       0.00296
    X0               0.00252       0.00258       0.00255       0.00257       0.00245
    Z0Z1             0.00886       0.00906       0.00833       0.00834       0.00829
    X0X1             0.00853       0.00880       0.00856       0.00851       0.00833
    Z0Z1Z2           0.02531       0.02810       0.02654       0.02804       0.02772
    E_TFIM           0.04646       0.05104       0.04841       0.05065       0.04988

  MSE, Haar average:
                 tetrahedron    octahedron          cube   icosahedron  dodecahedron
    Z0               0.00299       0.00295       0.00295       0.00293       0.00299
    X0               0.00293       0.00293       0.00297       0.00295       0.0029

In [12]:
# === The exact landscape: the theorem the Monte Carlo illustrates ===

# ---------------------------------------------------------------------------
# Exact landscape: the theorem behind Experiment 1 (no sampling, no RNG)
# ---------------------------------------------------------------------------
K_BODY = {"Z0": 1, "X0": 1, "Z0Z1": 2, "X0X1": 2, "Z0Z1Z2": 3}


def exact_backbone(povms, states, haar, exp1, exp1_haar):
    """Exact single-shot variances at the canonical dual, and the proposition.

    With the canonical dual, a site both terms touch contributes the operator
    sum_k (3 n_a)(3 n_b) E_k = 3 delta_ab Id + (9/V sum_k n_a n_b n_c) sigma_c,
    a site one term touches contributes exactly sigma_a (frame condition), so
    the variance sees the vertices only through moments of order <= 3: order 1
    vanishes (centered), order 2 is universal (2-design), order 3 vanishes for
    antipodal vertex sets.  Hence octahedron, cube, icosahedron, dodecahedron
    have identical variance on every (state, observable); a single k-body
    Pauli string sits at exactly 3^k - <P>^2 for all five solids (the
    tetrahedron's n_a^2 = 1/3 kills its same-letter third moments); and the
    tetrahedron can deviate only where a term pair shares a site with two
    different letters -- here only the TFIM energy, and only on states with
    the right correlations (the Haar states and the product state; TFIM and
    GHZ have real amplitudes, which zero the relevant <..sigma_y..> terms).
    The Experiment-1 Monte Carlo is then asserted to sit within 5 SE of the
    exact values it merely illustrates.
    """
    print("\nExact landscape at the canonical dual (deterministic):")
    obs_names = list(OBSERVABLES)
    state_items = list(states.items()) \
        + [(f"haar{i}", rho) for i, rho in enumerate(haar)]
    var = {}
    for pname, povm in povms.items():
        lut = lut_canonical(povm["s"])
        for sname, rho in state_items:
            p = born_tensor(rho, povm["E"])
            for on in obs_names:
                truth = exact_value(rho, OBSERVABLES[on])
                var[(pname, sname, on)] = \
                    exact_second_moment(p, OBSERVABLES[on], lut) - truth ** 2

    worst_anti = max(
        max(var[(pn, sn, on)] for pn in ANTIPODAL)
        - min(var[(pn, sn, on)] for pn in ANTIPODAL)
        for sn, _ in state_items for on in obs_names)
    assert worst_anti < 1e-10
    print(f"  antipodal solids identical on every (state, observable): "
          f"max spread = {worst_anti:.1e}: OK")

    worst_3k = max(
        abs(var[(pn, sn, on)] - (3.0 ** k - exact_value(rho, OBSERVABLES[on]) ** 2))
        for sn, rho in state_items for on, k in K_BODY.items() for pn in SOLIDS)
    assert worst_3k < 1e-10
    print(f"  single k-body strings: Var = 3^k - <P>^2 exactly, all five "
          f"solids: max |diff| = {worst_3k:.1e}: OK")

    dev = {sn: var[("tetrahedron", sn, "E_TFIM")]
           - var[("octahedron", sn, "E_TFIM")] for sn, _ in state_items}
    assert abs(dev["TFIM"]) < 1e-10 and abs(dev["GHZ"]) < 1e-10
    haar_devs = [dev[f"haar{i}"] for i in range(N_HAAR)]
    print("  tetrahedron deviations (third moment), E_TFIM only: "
          f"TFIM/GHZ = 0, product = {dev['product']:+.4f}, "
          f"Haar in [{min(haar_devs):+.4f}, {max(haar_devs):+.4f}]")
    print(f"    e.g. haar0: {var[('tetrahedron', 'haar0', 'E_TFIM')]:.4f} "
          f"(tetra) vs {var[('octahedron', 'haar0', 'E_TFIM')]:.4f} (others)")

    for sn in states:
        for on in obs_names:
            for pn in SOLIDS:
                st = exp1[(pn, sn, on)]
                assert abs(st["mse"] - var[(pn, sn, on)] / T_MAIN) \
                    < 5 * st["se_mse"] + 1e-12, (pn, sn, on)
    for on in obs_names:
        for pn in SOLIDS:
            ex = np.mean([var[(pn, f"haar{i}", on)]
                          for i in range(N_HAAR)]) / T_MAIN
            assert abs(exp1_haar[(pn, on)] - ex) / ex < 0.05, (pn, on)
    print("  Experiment-1 Monte Carlo within 5 SE of the exact values "
          "(TFIM/GHZ/product cells; Haar means within 5%): OK")

    out = {}
    for sn in list(states) + ["haar_mean"]:
        for on in obs_names:
            if sn == "haar_mean":
                uni = np.mean([var[("octahedron", f"haar{i}", on)]
                               for i in range(N_HAAR)])
                tet = np.mean([var[("tetrahedron", f"haar{i}", on)]
                               for i in range(N_HAAR)])
            else:
                uni, tet = var[("octahedron", sn, on)], \
                    var[("tetrahedron", sn, on)]
            out[f"exact/var/{sn}/{on}"] = np.array([uni])
            out[f"exact/var_tetra/{sn}/{on}"] = np.array([tet])
    return var, out


var_exact, exact_out = exact_backbone(povms, states, haar, exp1, exp1_haar)


Exact landscape at the canonical dual (deterministic):


  antipodal solids identical on every (state, observable): max spread = 7.8e-13: OK
  single k-body strings: Var = 3^k - <P>^2 exactly, all five solids: max |diff| = 3.6e-13: OK
  tetrahedron deviations (third moment), E_TFIM only: TFIM/GHZ = 0, product = -0.0000, Haar in [-2.2348, +5.7881]
    e.g. haar0: 40.3702 (tetra) vs 40.9394 (others)
  Experiment-1 Monte Carlo within 5 SE of the exact values (TFIM/GHZ/product cells; Haar means within 5%): OK


## 4. The fourth-moment ladder — where the solid choice reappears

One moment order up, the landscape stops being flat. For a single Pauli letter the single-shot
estimate is $3(\hat n_k)_a$, so
$\mathbb E[\hat o^4] = 81\cdot\frac1V\sum_k(\hat n_k)_a^4$ — and the state-dependent part is a
*fifth*-order odd moment, killed by antipodality (or by $(\hat n)_a^4 = \frac19$ for the
tetrahedron). So the fourth moment is **state-independent** and genuinely solid-dependent:

$$\mathbb E[\hat o^4] = 9 \;(\text{tetrahedron, cube}), \qquad 27 \;(\text{octahedron}), \qquad
\tfrac{81}{5} = 16.2 \;(\text{icosahedron, dodecahedron}).$$

The 5-designs sit *pinned at the uniform-sphere value in any orientation* — design strength fixes
the tails at the sphere's own weight; it does not minimize them. The extremes are worth feeling
physically: the cube's vertices all have $|(\hat n)_a| = \frac1{\sqrt3}$, so measuring a letter
returns $\pm\sqrt3$ *always* — the lightest possible tails at variance 3 — while the octahedron
(= standard Pauli-6 shadows) returns $0$ with probability exactly $\frac23$ **on any state** (its
equator is centered) and a $\pm3$ spike otherwise. Same mean, same variance, very different tails
— a reason to prefer the cube that no variance table can show, and the demo after the check makes
the distributions explicit.

In [13]:
# === The ladder, exactly ===

# ---------------------------------------------------------------------------
# Fourth-moment ladder: where the solid choice reappears, exactly
# ---------------------------------------------------------------------------
FOURTH_EXPECTED = {"tetrahedron": 9.0, "cube": 9.0, "octahedron": 27.0,
                   "icosahedron": 81.0 / 5.0, "dodecahedron": 81.0 / 5.0}


def fourth_moment_ladder(povms, states, haar):
    """Exact single-shot fourth moments of the canonical estimator.

    For a single Pauli sigma_a the single-shot estimate is f = 3 n_a, so
    E[f^4] = 81 (1/V) sum_k n_a^4 + 81 (1/V) sum_k n_a^4 (n_k . r).  The
    state-dependent term is a fifth-order odd moment and vanishes for every
    solid (antipodality; for the tetrahedron n_a^4 = 1/9 is constant), so
    E[f^4] is state-independent: 9 for tetrahedron and cube, 27 for the
    octahedron, 81/5 for icosahedron and dodecahedron -- whose 5-design
    strength pins them at the spherical value.  Design strength fixes the
    tails; it does not minimize them (Pauli-6/octahedron is heaviest, the
    cube lightest).
    """
    print("\nFourth-moment ladder: E[o^4] of the single-shot canonical "
          "estimator, single Paulis")
    state_items = list(states.items()) \
        + [(f"haar{i}", rho) for i, rho in enumerate(haar)]
    out = {}
    worst = 0.0
    for pname, povm in povms.items():
        s, V = povm["s"], povm["V"]
        vals = 81.0 * (s ** 4).mean(axis=0)
        assert vals.max() - vals.min() < 1e-12, pname
        assert abs(vals[0] - FOURTH_EXPECTED[pname]) < 1e-12, pname
        for sname, rho in state_items:      # state independence, exactly
            for r0 in site_bloch(rho):
                q = (1.0 + s @ r0) / V
                for a in range(3):
                    worst = max(worst, abs(q @ (3.0 * s[:, a]) ** 4 - vals[a]))
        out[f"exact4/{pname}"] = np.array([vals[0]])
        print(f"  {pname:12s} E[o^4] = {vals[0]:7.4f}   "
              f"(= 81/V sum_k n_a^4; every axis, site, state)")
    assert worst < 1e-12
    print(f"  state-independence across the full suite: "
          f"max |diff| = {worst:.1e}: OK")
    return out


four_out = fourth_moment_ladder(povms, states, haar)


Fourth-moment ladder: E[o^4] of the single-shot canonical estimator, single Paulis
  tetrahedron  E[o^4] =  9.0000   (= 81/V sum_k n_a^4; every axis, site, state)
  octahedron   E[o^4] = 27.0000   (= 81/V sum_k n_a^4; every axis, site, state)
  cube         E[o^4] =  9.0000   (= 81/V sum_k n_a^4; every axis, site, state)
  icosahedron  E[o^4] = 16.2000   (= 81/V sum_k n_a^4; every axis, site, state)
  dodecahedron E[o^4] = 16.2000   (= 81/V sum_k n_a^4; every axis, site, state)
  state-independence across the full suite: max |diff| = 7.1e-15: OK


In [14]:
# === Demo: same variance, different tails (local rng only) ===

rng_demo = np.random.default_rng(23)
r = rng_demo.standard_normal(3)
r *= rng_demo.random() / np.linalg.norm(r)          # a random state in the ball
for name in ("cube", "octahedron", "icosahedron"):
    s = povms[name]["s"]
    q = (1 + s @ r) / len(s)                        # Born probabilities
    vals = 3.0 * s[:, 2]                            # per-shot values, letter Z
    uq, inv = np.unique(np.round(vals, 9), return_inverse=True)
    probs = np.bincount(inv, weights=q)
    dist = ",  ".join(f"{u:+.3f} w.p. {p:.4f}" for u, p in zip(uq, probs))
    print(f"  {name:12s} {dist}")
    print(f"  {'':12s} E[o] = {q @ vals:+.4f}   "
          f"Var = {q @ vals ** 2 - (q @ vals) ** 2:.4f}   "
          f"E[o^4] = {q @ vals ** 4:.4f}")
print("\nthe octahedron's 0 has probability exactly 2/3 on ANY state "
      "(its equator sums to zero);\nthe cube never leaves +-sqrt(3)")

  cube         -1.732 w.p. 0.5032,  +1.732 w.p. 0.4968
               E[o] = -0.0110   Var = 2.9999   E[o^4] = 9.0000
  octahedron   -3.000 w.p. 0.1685,  +0.000 w.p. 0.6667,  +3.000 w.p. 0.1648
               E[o] = -0.0110   Var = 2.9999   E[o^4] = 27.0000
  icosahedron  -2.552 w.p. 0.1682,  -1.577 w.p. 0.1676,  +0.000 w.p. 0.3333,  +1.577 w.p. 0.1657,  +2.552 w.p. 0.1651
               E[o] = -0.0110   Var = 2.9999   E[o^4] = 16.2000

the octahedron's 0 has probability exactly 2/3 on ANY state (its equator sums to zero);
the cube never leaves +-sqrt(3)


## 5. Experiment 2 — optimizing the dual

**The job:** the canonical dual wasted none of the solids' redundancy — but the $(4V-16)$-dimensional
dual family is still a resource. Minimize the variance over it and see what overcompleteness buys.

The optimization runs **per Pauli letter**: the estimator only ever reads one letter per site, its
mean is pinned by the frame condition, so its second moment — a quadratic form in the dual's
coordinates, weighted by the outcome distribution — is the whole objective. The training
distribution enters that objective *only through the mean Bloch vector* of its reduced states
(the objective is linear in the distribution); for the TFIM sweep $h \in \{0.3,\dots,1.5\}$ plus
GHZ that mean points along $x$. Three levels:

- **canonical** — the baseline;
- **observable-optimized** — trained on the TFIM+GHZ distribution; a deployable protocol;
- **oracle** — handed the true reduced state of each site, per site. A *ceiling, not a protocol*:
  it can overfit the dual to the state being measured (watch the product-state $X_0$ cell reach
  exactly zero variance — a degenerate dual perfectly aligned with a pure $\ket{+}$ site).

Two sanity anchors before trusting any gains: feeding the QP the *uniform* distribution returns
the canonical dual for every solid (Haar-optimality — so gains live entirely in the prior), and
the tetrahedron's ratios are $1$ identically (nowhere to move).

In [15]:
# === Experiment 2: training, per-letter duals, Monte Carlo ===

def training_bloch_mean():
    """Mean Bloch vector of the training reduced states (TFIM sweep + GHZ).

    The QP objective is linear in the training distribution, which for
    effects (1/V)(Id + s.sigma) depends on the reduced states only through
    their mean Bloch vector.
    """
    vecs = [site_bloch(density(tfim_ground_state(g))) for g in TRAIN_G]
    ghz = np.zeros(16, dtype=complex)
    ghz[0] = ghz[15] = 1 / np.sqrt(2)
    vecs.append(site_bloch(density(ghz)))           # = 0 (maximally mixed)
    return np.concatenate(vecs).mean(axis=0)


def letter_duals(s, rbar, s_eff=None):
    """Optimized dual per Pauli letter for training distribution rbar."""
    if s_eff is None:
        s_eff = s
    pbar = (1.0 + s @ rbar) / len(s)
    return {a: optimize_dual(s_eff, pbar, a)[1] for a in range(3)}


def oracle_lut(s, site_r, s_eff=None, noisy_site_r=None):
    """Per-(site, letter) duals fed the true reduced state of each site."""
    if s_eff is None:
        s_eff = s
    if noisy_site_r is None:
        noisy_site_r = site_r
    V = len(s)
    lut = np.empty((N_QUBITS, 3, V))
    for i in range(N_QUBITS):
        pbar = (1.0 + s @ noisy_site_r[i]) / V
        for a in range(3):
            _, beta = optimize_dual(s_eff, pbar, a)
            lut[i, a] = 2.0 * beta[:, a]
    return lut


def experiment_2(povms, states, rng):
    print("\nExperiment 2: dual-frame optimization, noiseless "
          f"(shots={T_EXP2}, reps={R_REPS})")
    rbar = training_bloch_mean()
    print(f"  training-set mean Bloch vector: "
          f"({rbar[0]:.4f}, {rbar[1]:.4f}, {rbar[2]:.4f})")
    obs_names = ["Z0", "X0", "Z0Z1", "E_TFIM"]
    eval_states = {"TFIM": states["TFIM"], "product": states["product"]}

    # Haar-optimality: uniform training distribution recovers the canonical
    # dual (overcomplete solids), and the tetrahedron has no freedom at all.
    for pname, povm in povms.items():
        s, V = povm["s"], povm["V"]
        for a in range(3):
            _, beta = optimize_dual(s, np.full(V, 1 / V), a)
            assert np.abs(beta[:, a] - 1.5 * s[:, a]).max() < 1e-8, (pname, a)
    print("  Haar-optimality: uniform-distribution QP returns the canonical "
          "dual (all solids): OK")

    results = {}
    for pname, povm in povms.items():
        s = povm["s"]
        luts = {"canonical": lut_canonical(s),
                "observable": lut_from_letter_duals(letter_duals(s, rbar))}
        if pname == "tetrahedron":     # V = d^2: the dual family is a point
            assert np.abs(luts["observable"] - luts["canonical"]).max() < 1e-8
        for braw in letter_duals(s, rbar).values():
            assert dual_frame_residual(0.5 * np.ones(len(s)), braw, s) < 1e-9
        for sname, rho in eval_states.items():
            luts["oracle"] = oracle_lut(s, site_bloch(rho))
            p = born_tensor(rho, povm["E"])
            out = sample_outcomes(p, R_REPS * T_EXP2, rng)
            for on in obs_names:
                truth = exact_value(rho, OBSERVABLES[on])
                for lname, lut in luts.items():
                    em = exact_estimator_mean(p, OBSERVABLES[on], lut)
                    assert abs(em - truth) < 1e-9, (pname, sname, on, lname)
                    results[(pname, sname, on, lname)] = rep_stats(
                        shot_estimates(out, OBSERVABLES[on], lut), truth)
    print("  exact unbiasedness of optimized duals: OK (deterministic)")

    for on in ["Z0", "Z0Z1"]:   # tetrahedron control: ratios exactly 1
        r = results[("tetrahedron", "TFIM", on, "oracle")]["mse"] \
            / results[("tetrahedron", "TFIM", on, "canonical")]["mse"]
        assert abs(r - 1.0) < 1e-12
    print("  tetrahedron control: MSE ratio = 1 exactly: OK")

    for sname in eval_states:
        label = "TFIM h=1 (in-distribution)" if sname == "TFIM" \
            else "product state (out-of-distribution)"
        print(f"\n  MSE ratio to canonical, {label}:")
        print("    " + f"{'POVM':14s}{'level':12s}"
              + "".join(f"{on:>12s}" for on in obs_names))
        for pname in SOLIDS:
            for lname in ["observable", "oracle"]:
                row = []
                for on in obs_names:
                    a = results[(pname, sname, on, lname)]
                    c = results[(pname, sname, on, "canonical")]
                    ratio = a["mse"] / c["mse"]
                    se = ratio * np.hypot(a["se_mse"] / a["mse"],
                                          c["se_mse"] / c["mse"])
                    row.append(f"{ratio:7.2f} ({se:.2f})")
                print(f"    {pname:14s}{lname:12s}" + "".join(
                    f"{r:>12s}" for r in row))
    print("  (parenthesised: delta-method standard error; shots are shared "
          "across levels, so these are conservative)")
    return results


exp2 = experiment_2(povms, states, rng)        # [stream 3/5]


Experiment 2: dual-frame optimization, noiseless (shots=5000, reps=2000)
  training-set mean Bloch vector: (0.3992, 0.0000, 0.0000)
  Haar-optimality: uniform-distribution QP returns the canonical dual (all solids): OK


  exact unbiasedness of optimized duals: OK (deterministic)
  tetrahedron control: MSE ratio = 1 exactly: OK

  MSE ratio to canonical, TFIM h=1 (in-distribution):
    POVM          level                 Z0          X0        Z0Z1      E_TFIM
    tetrahedron   observable     1.00 (0.05)   1.00 (0.05)   1.00 (0.04)   1.00 (0.04)
    tetrahedron   oracle         1.00 (0.05)   1.00 (0.05)   1.00 (0.04)   1.00 (0.04)
    octahedron    observable     1.00 (0.04)   0.69 (0.03)   1.00 (0.04)   1.02 (0.05)
    octahedron    oracle         1.00 (0.04)   0.63 (0.03)   1.00 (0.04)   1.06 (0.05)
    cube          observable     0.88 (0.04)   1.00 (0.05)   0.76 (0.03)   0.74 (0.03)
    cube          oracle         0.87 (0.04)   1.00 (0.05)   0.74 (0.03)   0.67 (0.03)
    icosahedron   observable     0.93 (0.04)   0.89 (0.04)   0.86 (0.04)   0.86 (0.04)
    icosahedron   oracle         0.92 (0.04)   0.86 (0.04)   0.85 (0.04)   0.85 (0.04)
    dodecahedron  observable     0.92 (0.04)   0.89 (0.04)   

### The exact ratios, and the two cells worth staring at

The trained duals are deterministic functions of the training data, so every table entry is an
**exact** variance ratio via the outcome tensors; the Monte Carlo is merely asserted to agree
within five standard errors.

**The gains do not scale with the dual dimension.** On the in-distribution energy the *cube*
(16 free parameters) beats both 5-designs — the icosahedron (32) and the dodecahedron (64) — at
both levels. A quarter to a third off, at best, from redundancy that grew eightfold: the dual
family is a rapidly saturating resource.

**The backfire cell.** The octahedron's per-letter *oracle* on the TFIM energy lands at ratio
$1.069$ — *worse than canonical*, exactly. No contradiction: the per-letter objective minimizes
each letter's own second moment, but a composite observable's variance also carries **cross-term
covariances the objective never sees**, and on the octahedron the letter-optimal duals land on
the wrong side of them. Korhonen et al. construct two-qubit instances of exactly this; in the
Platonic family it arrives uninvited, as a computed number.

*Orientation note.* These gains are properties of the atlas orientation (they depend on how the
solid sits relative to the states being measured): re-pose a solid and its number moves, the
icosahedron's $\sim$15% included. What is orientation-invariant is the structure: convergence
of the gain, the saturation of the family, and §10's class ceiling.

In [16]:
# === Exact Experiment-2 ratios (deterministic duals -> exact table) ===

def exact_exp2_ratios(povms, states, exp2):
    """Exact MSE ratios for every Experiment-2 cell (no sampling, no RNG).

    The optimized duals are fixed functions of the training data, so each
    cell's single-shot variance -- hence each ratio to canonical -- is an
    exact number via exact_second_moment.  The Monte Carlo of Experiment 2
    is asserted to agree within 5 delta-method SE (conservative: all levels
    share the same shots).  Two cells worth naming: the octahedron's oracle
    X-dual on the product state has *exactly* zero variance (ratio 0), and
    the octahedron's per-letter oracle on the TFIM energy is exactly *worse*
    than canonical (ratio > 1) -- per-letter optimization minimizes each
    letter's own second moment, not the cross-term covariances a composite
    observable also carries.
    """
    print("\nExact Experiment-2 ratios (deterministic duals):")
    rbar = training_bloch_mean()
    obs_names = ["Z0", "X0", "Z0Z1", "E_TFIM"]
    eval_states = {"TFIM": states["TFIM"], "product": states["product"]}
    out = {}
    worst_z = 0.0
    for pname, povm in povms.items():
        s = povm["s"]
        luts = {"canonical": lut_canonical(s),
                "observable": lut_from_letter_duals(letter_duals(s, rbar))}
        for sname, rho in eval_states.items():
            luts["oracle"] = oracle_lut(s, site_bloch(rho))
            p = born_tensor(rho, povm["E"])
            for on in obs_names:
                truth = exact_value(rho, OBSERVABLES[on])
                v = {ln: exact_second_moment(p, OBSERVABLES[on], lut)
                     - truth ** 2 for ln, lut in luts.items()}
                for ln in ("observable", "oracle"):
                    exact_r = v[ln] / v["canonical"]
                    out[f"exact_ratio/{pname}/{sname}/{on}/{ln}"] = \
                        np.array([exact_r])
                    if pname == "tetrahedron":     # unique dual: ratio is 1
                        assert abs(exact_r - 1.0) < 1e-7, (sname, on, ln)
                    a = exp2[(pname, sname, on, ln)]
                    c = exp2[(pname, sname, on, "canonical")]
                    if exact_r < 1e-9:             # exact zero-variance cell
                        assert a["mse"] < 1e-9 * c["mse"], (pname, sname, on)
                        continue
                    mc_r = a["mse"] / c["mse"]
                    se = mc_r * np.hypot(a["se_mse"] / a["mse"],
                                         c["se_mse"] / c["mse"])
                    worst_z = max(worst_z, abs(mc_r - exact_r) / se)
    assert worst_z < 5.0
    print(f"  MC ratios within 5 SE of exact on all cells "
          f"(worst |z| = {worst_z:.2f}): OK")
    print("  tetrahedron control: exact ratio = 1 (unique dual): OK")
    assert out["exact_ratio/octahedron/product/X0/oracle"][0] < 1e-9
    korh = out["exact_ratio/octahedron/TFIM/E_TFIM/oracle"][0]
    assert korh > 1.0
    print(f"  octahedron X0 / product / oracle: exactly zero variance: OK")
    print(f"  octahedron E_TFIM / TFIM / oracle: exact ratio = {korh:.3f} > 1 "
          "(per-letter optimization can lose on composite observables)")
    return out


ratio_out = exact_exp2_ratios(povms, states, exp2)


Exact Experiment-2 ratios (deterministic duals):


  MC ratios within 5 SE of exact on all cells (worst |z| = 1.70): OK
  tetrahedron control: exact ratio = 1 (unique dual): OK
  octahedron X0 / product / oracle: exactly zero variance: OK
  octahedron E_TFIM / TFIM / oracle: exact ratio = 1.069 > 1 (per-letter optimization can lose on composite observables)


## 6. Experiment 3 — robust calibration

**The job:** turn on per-qubit depolarizing noise at rate $p$ and run the robust protocol:
calibrate the effective shrinkage on $\ket{0}$, then invert $\hat\eta$ instead of $\frac13$.

**What the calibration learns is an exact identity.** On $\ket{0}$ the mean sampled vertex's
$z$-component is
$\mathbb E[(\hat n_k)_z] = \frac1V\sum_k (\hat n_k)_z + \frac{1-p}{V}\sum_k(\hat n_k)_z^2 =
0 + \frac{1-p}3$ — the zero-sum and covariance identities again. One matvec, asserted for every
rate. Equally deterministic is the damage when you *don't* calibrate: the noiseless dual on noisy
outcomes shrinks every weight-$w$ term by exactly $(1-p)^w$ — a bias, so **no number of shots
helps**.

**Calibration is insurance.** It converts that bias into variance. The premium is the sampling
error of $\hat\eta$ itself, which the division writes multiplicatively into every estimate — at
$p = 0$ the robust estimator pays about **60% extra MSE before there is any noise to correct**
(0.050 → 0.080 on the energy). The premium is generated by $\operatorname{Var}[\hat\eta] \propto
1/(4R_C)$ and amortizes accordingly; past the crossover near $p \approx 0.02$ the trade wins at
every rate. Re-optimizing the dual *on the calibrated effects* (same QP, now fed
$\hat\eta$-rescaled vertices) claws back a further $\sim$7% at $p = 0.1$ — optimization survives
noise, but shrinks.

In [17]:
# === Experiment 3: calibration + the estimator hierarchy under noise ===

def calibrate(s, T, t, n_reps, n_shots, rng):
    """|0> calibration: per-rep mean sampled vertex (3-vector); eta = z part."""
    q = single_qubit_probs(s, T, t, np.array([0.0, 0.0, 1.0]))
    cum = np.cumsum(q)
    cum[-1] = 1.0
    idx = np.searchsorted(cum, rng.random((n_reps, n_shots * N_QUBITS)),
                          side="right")
    return s[idx].mean(axis=1)        # (n_reps, 3)


def experiment_3(povms, rho, rng):
    print("\nExperiment 3: robust calibration, icosahedron, TFIM h=1 "
          f"(shots={T_MAIN}, reps={R_REPS}, R_C={RC_CAL}, depolarizing sweep)")
    povm = povms["icosahedron"]
    s, V = povm["s"], povm["V"]
    obs_names = ["E_TFIM", "Z0Z1", "X0"]
    truths = {on: exact_value(rho, OBSERVABLES[on]) for on in obs_names}
    rbar = training_bloch_mean()
    site_r = site_bloch(rho)
    levels = ["noiseless", "robust", "optimized", "oracle"]
    results = {}

    # the calibration target is an exact identity, not a statistical fact:
    # E[eta_hat] = z-component of E[sampled vertex] = (1-p)/3, one matvec
    zhat = np.array([0.0, 0.0, 1.0])
    for p_rate in DEPOL_RATES:
        v = single_qubit_probs(s, *chan_depolarizing(p_rate), zhat) @ s
        assert abs(v[2] - (1 - p_rate) / 3) < 1e-12, p_rate
        assert np.hypot(v[0], v[1]) < 1e-12, p_rate
    print("  exact identity E[eta_hat] = (1-p)/3 (and E[v_perp] = 0), "
          "every rate: OK")

    for p_rate in DEPOL_RATES:
        T3, t3 = chan_depolarizing(p_rate)
        pt = born_tensor(rho, noisy_effects(s, T3, t3))
        out = sample_outcomes(pt, R_REPS * T_MAIN, rng)
        eta_hat = calibrate(s, T3, t3, R_REPS, RC_CAL, rng)[:, 2]

        # deterministic checks: eta target and the (1-p)^k bias factor of
        # the noiseless dual on noisy outcomes
        eta_target = (1 - p_rate) / 3
        lut0 = lut_canonical(s)
        for on, k_body in [("Z0", 1), ("Z0Z1", 2), ("E_TFIM", None)]:
            em = exact_estimator_mean(pt, OBSERVABLES[on], lut0)
            if k_body is not None:
                truth = exact_value(rho, OBSERVABLES[on])
                assert abs(em - (1 - p_rate) ** k_body * truth) < 1e-9

        est = {(on, ln): np.empty(R_REPS * T_MAIN)
               for on in obs_names for ln in levels}
        for r in range(R_REPS):
            sl = slice(r * T_MAIN, (r + 1) * T_MAIN)
            eta = eta_hat[r]
            s_eff = 3 * eta * s                # depolarizing-model effects
            luts = {
                "noiseless": lut0,
                "robust": lut_robust_canonical(s, eta),
                "optimized": lut_from_letter_duals(
                    letter_duals(s, 3 * eta * rbar, s_eff=s_eff)),
                "oracle": oracle_lut(s, site_r, s_eff=s_eff,
                                     noisy_site_r=(1 - p_rate) * site_r),
            }
            for on in obs_names:
                for ln in levels:
                    est[(on, ln)][sl] = shot_estimates(
                        out[sl], OBSERVABLES[on], luts[ln])
        for on in obs_names:
            for ln in levels:
                results[(p_rate, on, ln)] = rep_stats(est[(on, ln)],
                                                      truths[on])
        results[(p_rate, "eta")] = (eta_hat.mean(), eta_target)

    # calibration convergence at large R_C
    for p_rate in DEPOL_RATES:
        T3, t3 = chan_depolarizing(p_rate)
        eta = calibrate(s, T3, t3, 1, 5000, rng)[0, 2]
        se = 1 / np.sqrt(3 * 5000 * N_QUBITS)   # Var[s_z] <= 1/3
        assert abs(eta - (1 - p_rate) / 3) < 5 * se, p_rate
    print("  calibration convergence (R_C=5000): "
          "|eta_hat - (1-p)/3| < 5 SE for all rates: OK")
    print("  noiseless-dual bias factor (1-p)^k: exact (deterministic): OK")

    for p_rate in DEPOL_RATES:
        m, tgt = results[(p_rate, "eta")]
        print(f"    p={p_rate:<5} eta_hat = {m:.4f}  target = {tgt:.4f}")

    print(f"\n  bias^2 / variance / MSE at p=0.1 (icosahedron):")
    for on in ["E_TFIM", "Z0Z1"]:
        print(f"    {on}  (truth {truths[on]:+.4f})")
        for ln in levels:
            st = results[(0.1, on, ln)]
            print(f"      {ln:12s} bias2 {st['bias2']:9.5f}   "
                  f"var {st['var']:9.5f}   mse {st['mse']:9.5f}")

    print("\n  MSE for E_TFIM across the sweep:")
    print("    " + f"{'level':14s}"
          + "".join(f"{f'p={p}':>10s}" for p in DEPOL_RATES))
    for ln in levels:
        print(f"    {ln:14s}" + "".join(
            f"{results[(p, 'E_TFIM', ln)]['mse']:10.4f}"
            for p in DEPOL_RATES))

    # at p = 0 every level must be unbiased (bias^2 within MC error of 0)
    for on in obs_names:
        for ln in levels:
            st = results[(0.0, on, ln)]
            assert st["bias2"] < 25 * st["var"] / R_REPS, (on, ln)
    print("  p=0: all four levels unbiased within MC error: OK")
    return results


exp3 = experiment_3(povms, states["TFIM"], rng)        # [stream 4/5]


Experiment 3: robust calibration, icosahedron, TFIM h=1 (shots=1000, reps=2000, R_C=1000, depolarizing sweep)
  exact identity E[eta_hat] = (1-p)/3 (and E[v_perp] = 0), every rate: OK


  calibration convergence (R_C=5000): |eta_hat - (1-p)/3| < 5 SE for all rates: OK
  noiseless-dual bias factor (1-p)^k: exact (deterministic): OK
    p=0.0   eta_hat = 0.3335  target = 0.3333
    p=0.01  eta_hat = 0.3299  target = 0.3300
    p=0.05  eta_hat = 0.3170  target = 0.3167
    p=0.1   eta_hat = 0.3002  target = 0.3000
    p=0.2   eta_hat = 0.2664  target = 0.2667

  bias^2 / variance / MSE at p=0.1 (icosahedron):
    E_TFIM  (truth -5.2263)
      noiseless    bias2   0.57315   var   0.04859   mse   0.62171
      robust       bias2   0.00001   var   0.11111   mse   0.11106
      optimized    bias2   0.00000   var   0.10335   mse   0.10330
      oracle       bias2   0.00000   var   0.10313   mse   0.10308
    Z0Z1  (truth +0.6533)
      noiseless    bias2   0.01461   var   0.00889   mse   0.02350
      robust       bias2   0.00002   var   0.01469   mse   0.01470
      optimized    bias2   0.00001   var   0.01312   mse   0.01312
      oracle       bias2   0.00001   var   0.0128

## 7. The blind spot, and the free diagnostic

**The geometric one-liner:** the calibration probes the channel through a single state, so any
channel that **fixes $\ket{0}$** — dephasing and amplitude damping both do — is invisible to it:
$\hat\eta$ stays pinned at $\frac13$, reading "noiseless", and the robust estimator silently
reverts to the canonical one, reporting the *noisy* expectation values as though clean.

What it reports is itself exactly predictable: pushing the channel through the estimator, the
reported value of $\sigma_a$ on a site with Bloch vector $r$ is $(Tr + t)_a$, a residual bias of
$(Tr + t - r)_a$ — the cell asserts the measured biases sit on that prediction at every rate.
**Silence, not error, is the failure mode**: the wrong number ships under a clean confidence
interval, and no statistics on the estimation side can flag it.

**The free diagnostic.** The calibration shots already contain a full 3-vector: the mean sampled
vertex converges to $\hat v = \frac13(T\hat z + t)$, the calibrated image of the whole readout
axis. The protocol keeps its $z$-part ($\hat\eta$) and discards the rest — but the transverse
remainder $\hat v_\perp$ flags any channel that *tilts* the $z$-axis (a coherent misrotation
folded into readout, which is what drifting calibration produces): the tilted-depolarizing test
below reads $\hat\eta = 0.313$ — impersonating honest depolarizing noise at $p\approx0.06$ —
while $|\hat v_\perp| = 0.065$ busts it. The diagnostic's own blind spot: channels that fix the
whole $z$-axis *direction* — amplitude damping maps $\hat z \mapsto \hat z$ exactly, so it is
invisible to $\hat\eta$ **and** to $\hat v_\perp$. Narrower, at zero experimental cost; not
closed. Closing it structurally is §8's job.

In [18]:
# === Calibration blindness + the anisotropy diagnostic ===

def blindness(povms, rho, rng):
    print("\nCalibration blindness (icosahedron, robust-canonical estimator, "
          f"TFIM h=1, shots={T_MAIN}, reps={R_REPS})")
    povm = povms["icosahedron"]
    s = povm["s"]
    site_r = site_bloch(rho)
    channels = {"dephasing": chan_dephasing, "amp-damping": chan_amp_damping}
    zhat = np.array([0.0, 0.0, 1.0])
    results = {}

    for cname, chan in channels.items():
        for p_rate in BLIND_RATES:
            T3, t3 = chan(p_rate)
            # the channel fixes |0>, so calibration sees "noiseless": exact.
            # As in experiment_3 the target is an identity, not a statistical
            # fact -- E[eta_hat] is the z part of E[sampled vertex], pinned at
            # 1/3 at every rate -- so it is computed, one matvec, not narrated
            assert abs((T3 @ zhat + t3)[2] - 1.0) < 1e-12
            v_cal = single_qubit_probs(s, T3, t3, zhat) @ s
            assert abs(v_cal[2] - 1 / 3) < 1e-12, (cname, p_rate)
            assert np.hypot(v_cal[0], v_cal[1]) < 1e-12, (cname, p_rate)
            eta_hat = calibrate(s, T3, t3, R_REPS, RC_CAL, rng)[:, 2]
            # and the sampled mean meets it: R_REPS x RC_CAL x N_QUBITS draws
            # of s_z with Var[s_z] <= 1/3, so 5 SE = 1.0e-3, against a worst
            # measured deviation of 2.4e-4 over the eight rows
            se_eta = 1 / np.sqrt(3 * R_REPS * RC_CAL * N_QUBITS)
            assert abs(eta_hat.mean() - 1 / 3) < 5 * se_eta, (cname, p_rate)
            pt = born_tensor(rho, noisy_effects(s, T3, t3))
            out = sample_outcomes(pt, R_REPS * T_MAIN, rng)
            row = {"eta": eta_hat.mean()}
            for on, i, a in [("X0", 0, "X"), ("Z0", 0, "Z")]:
                obs = OBSERVABLES.get(on) or [(1.0, [(i, a)])]
                est = np.empty(R_REPS * T_MAIN)
                for r in range(R_REPS):
                    sl = slice(r * T_MAIN, (r + 1) * T_MAIN)
                    est[sl] = shot_estimates(
                        out[sl], obs, lut_robust_canonical(s, eta_hat[r]))
                bias = est.reshape(R_REPS, -1).mean(axis=1).mean() \
                    - site_r[i, AXIS[a]]
                # eta_hat -> 1/3, so the estimator silently reports the
                # *noisy* expectation: predicted bias = (T r + t - r)_a
                pred = (T3 @ site_r[i] + t3 - site_r[i])[AXIS[a]]
                assert abs(bias - pred) < 0.01, (cname, p_rate, on)
                row[on] = (bias, pred)
            results[(cname, p_rate)] = row

    print("    channel       p     eta_hat   bias X0 (pred)      bias Z0 (pred)")
    for (cname, p_rate), row in results.items():
        bx, px = row["X0"]
        bz, pz = row["Z0"]
        print(f"    {cname:12s}{p_rate:5.2f}   {row['eta']:.4f}   "
              f"{bx:+.4f} ({px:+.4f})   {bz:+.4f} ({pz:+.4f})")
    print("  eta_hat pinned at 1/3 (exact, and the sampled mean within 5 SE) "
          "while biases\n  match the Heisenberg-picture prediction: OK")

    print("\n  Anisotropy diagnostic: the calibration shots already contain "
          "the full 3-vector\n  v_hat -> (1/3) T z_hat; "
          "channels tilting the z-axis are flagged for free:")
    tests = {
        "depolarizing p=0.1": chan_depolarizing(0.1),
        "dephasing p=0.1": chan_dephasing(0.1),
        "amp-damping p=0.1": chan_amp_damping(0.1),
        "tilted-depol p=0.05, theta=0.2": chan_tilted_depolarizing(0.05, 0.2),
    }
    diag = {}
    for label, (T3, t3) in tests.items():
        v = calibrate(s, T3, t3, 1, 5000, rng)[0]
        v_exact = (T3 @ zhat + t3) / 3
        perp = np.hypot(v[0], v[1])
        diag[label] = (v, v_exact, perp)
        print(f"    {label:32s} eta_hat = {v[2]:7.4f}   "
              f"|v_perp| = {perp:.4f}   (exact {np.hypot(*v_exact[:2]):.4f})")
    assert diag["tilted-depol p=0.05, theta=0.2"][2] > 0.05
    assert all(diag[k][2] < 0.02 for k in list(tests)[:3])
    print("  z-axis-fixing channels: |v_perp| ~ 0; tilted channel flagged: OK")

    short = {"depolarizing p=0.1": "depol", "dephasing p=0.1": "dephasing",
             "amp-damping p=0.1": "ampdamp",
             "tilted-depol p=0.05, theta=0.2": "tilted"}
    aniso = {short[k]: np.array([diag[k][0][2], diag[k][2],
                                 np.hypot(*diag[k][1][:2])]) for k in tests}
    return results, aniso


blind, aniso = blindness(povms, states["TFIM"], rng)        # [stream 5/5]


Calibration blindness (icosahedron, robust-canonical estimator, TFIM h=1, shots=1000, reps=2000)


    channel       p     eta_hat   bias X0 (pred)      bias Z0 (pred)
    dephasing    0.01   0.3335   -0.0143 (-0.0131)   +0.0006 (+0.0000)
    dephasing    0.05   0.3332   -0.0644 (-0.0653)   +0.0005 (+0.0000)
    dephasing    0.10   0.3333   -0.1297 (-0.1307)   +0.0007 (+0.0000)
    dephasing    0.20   0.3333   -0.2595 (-0.2613)   +0.0014 (+0.0000)
    amp-damping  0.01   0.3331   -0.0007 (-0.0033)   +0.0091 (+0.0100)
    amp-damping  0.05   0.3333   -0.0178 (-0.0165)   +0.0487 (+0.0500)
    amp-damping  0.10   0.3332   -0.0331 (-0.0335)   +0.0990 (+0.1000)
    amp-damping  0.20   0.3335   -0.0706 (-0.0690)   +0.2007 (+0.2000)
  eta_hat pinned at 1/3 (exact, and the sampled mean within 5 SE) while biases
  match the Heisenberg-picture prediction: OK

  Anisotropy diagnostic: the calibration shots already contain the full 3-vector
  v_hat -> (1/3) T z_hat; channels tilting the z-axis are flagged for free:
    depolarizing p=0.1               eta_hat =  0.3006   |v_perp| = 0.0017   (ex

*The shared stream is now retired.* Everything below — the twirl checks, the two-protocol
channels, the gate-noise study, the scaling study — is deterministic: exact finite group
averages and exact linear algebra, asserted at $10^{-12}$.

## 8. The twirl: one lemma, two conjugations

**The lemma (the only one).** Let a finite rotation group $G \subset SO(3)$ act irreducibly on
$\mathbb R^3$ — no invariant line. Then for any matrix $X$ and any vector $u$:

$$\frac1{|G|}\sum_{g\in G} R_g^\top X\, R_g = \frac{\operatorname{tr}X}{3}\,\mathrm{Id},
\qquad\qquad \frac1{|G|}\sum_{g\in G} R_g^\top u = 0.$$

(The average commutes with every $R_h$ — re-index the sum — so Schur forces a scalar; the average
preserves the trace, which fixes the scalar. The vector average spans an invariant subspace,
which must be $\{0\}$.) The tetrahedral, octahedral and icosahedral rotation groups all act
irreducibly — which is why a $T$-draw can twirl at all.

**The five-liner, twice.** A protocol's *estimator channel* is the affine map $r \mapsto Mr + m$
from the state's Bloch vector to the mean sampled snapshot vertex (ideal: $r/3$).

*Randomized-projective* — draw $g$, apply $U_g$, apply the fixed alignment $A$ (vertex $v$ nearest
$+\hat z$ → $\hat z$), let the noise $r \to Tr+t$ act, measure $Z$:
1. outcome $(g, b)$ post-processes into the snapshot $b\,R_g^\top v$;
2. its probability is $\frac12\bigl(1 + b\,[T(AR_g r) + t]_z\bigr)$ — **the same $g$ sits in the
   probability and in the snapshot**;
3. summing over $b = \pm1$: $M = \mathbb E_g\, R_g^\top (vw^\top) R_g$ with
   $w = A^\top T^\top\hat z$ — the twirled object is the rank-one **composition**
   $\hat z\hat z^\top T$ of readout projector and noise, conjugated into the solid's frame;
4. the lemma: $M = \frac{\operatorname{tr}[\hat z\hat z^\top T]}{3}\,\mathrm{Id} =
   \frac{T_{zz}}{3}\,\mathrm{Id}$ — the trace picks out the noise's entry **along the readout
   axis**;
5. the offset $m = (\hat z\cdot t)\,\mathbb E_g R_g^\top v = 0$ — the vector average dies by
   irreducibility.

*Twirled-native* — draw $g$, apply $U_g$, run the untouched Naimark circuit, relabel by $g$:
1. outcome $(g, k)$ post-processes into $R_g^\top \hat n_k$;
2. its probability is $\frac1V\bigl(1 + \hat n_k\cdot(TR_g r + t)\bigr)$ — the readout ranges over
   all $V$ effects **whatever the draw**: nothing downstream of the noise depends on $g$;
3. summing over $k$ first contracts the vertex covariance
   $\sum_k \hat n_k\hat n_k^\top = \frac V3\,\mathrm{Id}$, leaving
   $M = \mathbb E_g\, R_g^\top \frac{T}{3} R_g$ — the twirled object is **the noise alone**;
4. the lemma: $M = \frac{\operatorname{tr}T/3}{3}\,\mathrm{Id}$ — the isotropic mean, which has
   forgotten every axis;
5. the offset dies **twice over**, and the two deaths are different mathematical facts: its POVM
   part rides on $\sum_k \hat n_k = 0$ — the *solid* is centered — and its noise-shift part on
   $\mathbb E_g R_g^\top = 0$ — the *group* leaves no invariant vector.

Step 3 never used antipodality, so the twirled-native column exists for **all five solids — the
tetrahedral SIC included**. Decomposability is a requirement of the projective route only.

**The physicist's version (no formulas).** The two protocols differ in exactly one place: *who
decides the readout axis.* In randomized-projective the same coin that rotates the state also
decides — through the fixed $\hat z$ readout — which lab-frame direction of the noise gets probed:
draw and readout are perfectly correlated, the correlation sits *inside* the average, and what
survives is the noise as seen along the readout axis, $T_{zz}$. In twirled-native the readout is
the same fixed POVM every shot; there is nothing for the draw to correlate with, the noise is
seen whole, and only its isotropic part survives, $\operatorname{tr}T/3$. That correlation is not
a nuisance to be argued away — it *is* the difference between the protocols.

### Boxes and wires: where the theorems reach

```text
native:                 rho ──[noise N]──[Naimark dilation]── outcome k
twirled-native:         rho ──[U_g]──[noise N]──[Naimark dilation]── k, relabel by g
randomized-projective:  rho ──[U_g]──[noise N]──[A]──[measure Z]── b, post-process by (g, b)
```

The theorems cover any noise $N$ that is **measurement-side** (between the drawn rotation and the
readout) and **independent of the draw** — for such $N$ the twirl is exact, whatever $N$ is. Not
covered: noise *inside the drawn word's own gates* (correlated with $g$ — that is §9, and Schur
has nothing to say about it), and state-preparation noise (that deforms $\rho$ itself: shadows
faithfully report the noisy state — it is not a measurement error at all). For twirled-native the
dilation's own noise is $g$-independent by construction — the dilation is the same circuit every
shot — so it sits inside the theorem's reach; the *model* of it (a pre-measurement channel on the
data qubit rather than an ancilla-side deformation) is a modeling choice, isolated in §9.

In [19]:
# === The twirl check: the projective channel is exactly depolarizing at T_zz/3 ===

def align_to_z(v):
    """Rotation A with A v = zhat (Rodrigues; v a unit vector)."""
    zhat = np.array([0.0, 0.0, 1.0])
    c, ax = v @ zhat, np.cross(v, zhat)
    s = np.linalg.norm(ax)
    if s < 1e-12:
        return np.eye(3) if c > 0 else np.diag([1.0, -1.0, -1.0])
    ax = ax / s
    K = np.array([[0.0, -ax[2], ax[1]], [ax[2], 0.0, -ax[0]],
                  [-ax[1], ax[0], 0.0]])
    return np.eye(3) + s * K + (1 - c) * (K @ K)


def twirl_check(povms):
    """Randomized-projective: the exact estimator channel is depolarizing.

    The protocol draws g uniformly from the covariance group, applies U_g,
    then one fixed alignment A taking a vertex axis v to the readout axis
    zhat (the octahedron already has zhat as a vertex; the icosahedron's
    zhat is an *edge* axis, so it genuinely needs one), lets the
    measurement-side noise r -> T r + t act, and measures the z basis;
    outcome (g, b) is post-processed into the snapshot vertex b R_g^T v.
    Summing probability x snapshot over (g, b) gives the estimator's
    channel exactly, and the twirled object is the *composition*
    zhat zhat^T T of readout projector and noise -- readout and conjugation
    share the same g -- not the noise alone.  Schur's lemma collapses it
    all the same: the two rotation groups act irreducibly on the Bloch
    space, so averaging any matrix M over conjugation gives (tr M / 3) Id
    and any vector averages to zero, but the trace picks out
    tr(zhat zhat^T T) = T_zz.  The channel is depolarizing with shrinkage
    eta = T_zz / 3, the noise's diagonal entry along the readout axis
    (Chen et al.'s single-qubit fidelity f), NOT (tr T / 3) / 3 -- that
    scalar twirls the noise alone and is the OTHER protocol's, twirled-
    native, derived in two_protocol_twirl below.  Dephasing along the
    readout axis is the eta = 1/3 case: twirled into no noise at all.
    The |0> calibration reads exactly this eta, so the robust inversion
    is exactly unbiased -- asserted on a random state.

    Checked to 1e-12 with the SO(3) groups from data/group_{O,I}.npz: the
    raw Schur lemma on a random matrix and vector (a probability-one
    witness of irreducibility, the twirl defect being linear in M), the
    orbit {+- R_g^T v} covering the solid's vertex set uniformly
    (randomized-projective samples the POVM it claims to), and the
    channel + calibration for dephasing, amplitude damping, and a random
    affine map.
    """
    print("\nTwirl check: randomized-projective's exact estimator "
          "channel\n(group-averaged over the atlas rotations, alignment "
          "included) is depolarizing:")
    rng = np.random.default_rng(11)
    zhat = np.array([0.0, 0.0, 1.0])
    tests = {
        "dephasing p=0.13": chan_dephasing(0.13),
        "amp-damping g=0.21": chan_amp_damping(0.21),
        "random affine": (rng.standard_normal((3, 3)) * 0.3,
                          rng.standard_normal(3) * 0.2),
    }
    for gname, solid in (("O", "octahedron"), ("I", "icosahedron")):
        R = np.load(DATA / f"group_{gname}.npz")["rotations"]

        # Schur lemma, raw: twirl(M) = (tr M / 3) Id, avg R^T u = 0
        M = rng.standard_normal((3, 3))
        u = rng.standard_normal(3)
        M_avg = np.einsum("gji,jk,gkl->il", R, M, R) / len(R)
        assert np.abs(M_avg - (np.trace(M) / 3) * np.eye(3)).max() < 1e-12
        assert np.abs(np.einsum("gji,j->i", R, u) / len(R)).max() < 1e-12

        # alignment: identify the readout axis with a vertex axis
        s = povms[solid]["s"]
        v = s[np.argmax(s[:, 2])]        # convention: first vertex nearest +z
        if gname == "O":
            assert np.abs(v - zhat).max() < 1e-12       # already aligned
        else:                            # unaligned z is not a vertex axis
            assert np.linalg.norm(s - zhat, axis=1).min() > 0.1
        A = align_to_z(v)

        # the orbit of the readout axis is the vertex set, uniformly
        orbit = np.einsum("gji,j->gi", R, v)            # R_g^T v = R_g^T A^T z
        orbit = np.concatenate([orbit, -orbit])
        counts = (np.linalg.norm(orbit[:, None, :] - s[None], axis=2)
                  < 1e-9).sum(axis=0)
        assert (counts == 2 * len(R) // len(s)).all(), gname
        print(f"  group {gname} ({len(R)} rotations): Schur lemma on random "
              f"(M, u): OK; orbit of the\n    readout axis = the {len(s)} "
              f"{solid} vertices, {counts[0]}x each: OK")

        for label, (T3, t3) in tests.items():
            # channel: E_g outer(R^T v, R^T w) with w = (T A)^T z; shift
            # carries the mean sampled vertex E_g[R^T v] = 0 (centered)
            w = A.T @ T3.T @ zhat
            ch = np.einsum("gji,j,gkl,k->il", R, v, R, w) / len(R)
            m = (zhat @ t3) * np.einsum("gji,j->i", R, v) / len(R)
            assert np.abs(ch - (T3[2, 2] / 3) * np.eye(3)).max() < 1e-12, \
                (gname, label)
            assert np.abs(m).max() < 1e-12, (gname, label)
            eta = (ch @ zhat + m)[2]     # what the |0> calibration reads
            r = rng.standard_normal(3) * 0.5
            assert np.abs((ch @ r + m) / eta - r).max() < 1e-12
            print(f"    {label:20s} eta = T_zz/3 = {eta:+.6f}   "
                  f"((tr T/3)/3 would read {np.trace(T3) / 9:+.6f})")
    print("  channel scalar + shift zero to 1e-12; calibration reads eta; "
          "inversion unbiased: OK")


twirl_check(povms)


Twirl check: randomized-projective's exact estimator channel
(group-averaged over the atlas rotations, alignment included) is depolarizing:
  group O (24 rotations): Schur lemma on random (M, u): OK; orbit of the
    readout axis = the 6 octahedron vertices, 8x each: OK
    dephasing p=0.13     eta = T_zz/3 = +0.333333   ((tr T/3)/3 would read +0.275556)
    amp-damping g=0.21   eta = T_zz/3 = +0.263333   ((tr T/3)/3 would read +0.285293)
    random affine        eta = T_zz/3 = +0.074689   ((tr T/3)/3 would read +0.016104)
  group I (60 rotations): Schur lemma on random (M, u): OK; orbit of the
    readout axis = the 12 icosahedron vertices, 10x each: OK
    dephasing p=0.13     eta = T_zz/3 = +0.333333   ((tr T/3)/3 would read +0.275556)
    amp-damping g=0.21   eta = T_zz/3 = +0.263333   ((tr T/3)/3 would read +0.285293)
    random affine        eta = T_zz/3 = +0.074689   ((tr T/3)/3 would read +0.016104)
  channel scalar + shift zero to 1e-12; calibration reads eta; inversion unbia

In [20]:
# === The two protocols, side by side, on every solid and both draws ===

# ---------------------------------------------------------------------------
# Two protocols: randomized-projective vs twirled-native, as exact channels
# ---------------------------------------------------------------------------
COVARIANCE_GROUP = {"tetrahedron": "T", "octahedron": "O", "cube": "O",
                    "icosahedron": "I", "dodecahedron": "I"}

# the generic probe noise of randomized_implementations.py, verbatim, so the
# two modules' exact scalars cross-check value for value: T_zz = 0.62,
# tr(T)/3 = 0.72 (genericity re-asserted in two_protocol_twirl)
T_PROBE = np.array([[0.83, 0.06, -0.11],
                    [-0.04, 0.71, 0.09],
                    [0.12, -0.07, 0.62]])

t_PROBE = np.array([0.05, -0.03, 0.17])


def channel_R1(s, R, T, t):
    """Exact estimator channel of the randomized-projective protocol.

    Draw g uniformly from the rotations R, apply U_g, apply the fixed
    alignment A (vertex nearest +z -> zhat), let the measurement-side noise
    r -> T r + t act, read out Z; outcome (g, b) is post-processed into the
    snapshot vertex b R_g^T v.  Returns (M, m) of the mean sampled vertex,
    E[vertex] = M r + m -- this module's convention (ideal channel Id/3;
    randomized_implementations.py includes the canonical dual's factor 3,
    so its channels are 3x these).  Same expressions as twirl_check().
    """
    zhat = np.array([0.0, 0.0, 1.0])
    v = s[np.argmax(s[:, 2])]
    A = align_to_z(v)
    w = A.T @ T.T @ zhat
    M = np.einsum("gji,j,gkl,k->il", R, v, R, w) / len(R)
    m = (zhat @ t) * np.einsum("gji,j->i", R, v) / len(R)
    return M, m


def channel_R2(s, R, T, t):
    """Exact estimator channel of the twirled-native protocol.

    Draw g, apply U_g, measure the NATIVE (Naimark) POVM -- all V effects
    -- and relabel by g in post-processing: snapshot vertex R_g^T n_k with
    probability (1/V)(1 + n_k . (T R_g r + t)).  Same mean-sampled-vertex
    convention as channel_R1; no alignment, no decomposability -- every
    solid admitted.  The vertex sums are kept explicit (rather than
    substituting the 2-design identity), so the computed channel is
    per-solid evidence, not a closed form restated.
    """
    C = s.T @ s / len(s)                     # = Id/3: the 2-design identity
    M = np.einsum("gji,jk,kl,glm->im", R, C, T, R) / len(R)
    m = np.einsum("gji,j->i", R, s.mean(axis=0) + C @ t) / len(R)
    return M, m


def orbit_hits(s, R):
    """How often the signed orbit {+-R_g^T v} of the alignment vertex hits
    each vertex.  Uniform counts = the draw's coin samples the POVM's
    vertex set uniformly (the projective route realizes the POVM); a zero
    means that vertex is never measured at all."""
    v = s[np.argmax(s[:, 2])]
    orbit = np.einsum("gji,j->gi", R, v)
    orbit = np.concatenate([orbit, -orbit])
    d = np.linalg.norm(orbit[:, None, :] - s[None], axis=2)
    assert (d.min(axis=1) < 1e-9).all()      # every draw lands on a vertex
    return (d < 1e-9).sum(axis=0)


def two_protocol_twirl(povms):
    """The two randomized implementations as exact estimator channels.

    Section 4.2.4 distinguishes two protocols that both carried the name
    "randomized implementation"; this check computes both on this module's
    estimator side, giving the appendix its own citable numbers (the
    symbolic gate for the underlying claims is
    randomized_implementations.py; the probe noise here is that module's
    verbatim, so the scalars cross-check value for value).

    Checked to 1e-12, per solid and per draw (covariance group and the
    uniform T draw): randomized-projective is exactly depolarizing at
    eta = T_zz/3 for the four antipodal solids -- the readout is drawn
    with the same g that conjugates the noise, so only the readout-axis
    entry survives -- and twirled-native at eta = (tr T/3)/3 for ALL FIVE,
    the tetrahedral SIC included; offsets die; the calibrated inversion is
    exactly unbiased; a local-rng random affine map re-witnesses each
    equality with probability one.  The T draw is the universal minimal
    twirl, and its orbit realizes octahedron/cube/icosahedron vertex-
    uniformly (x4/x3/x2) while covering only 6 of the dodecahedron's 10
    axes -- eight vertices never sampled, so the projective route's single
    draw does not realize the dodecahedral POVM there (the channel itself
    stays exactly depolarizing: it is a perfectly unbiased *different*
    measurement).  Payoff for the blindness study: dephasing and amplitude
    damping -- the fixers of |0> that pin the native calibration at 1/3 --
    twirl to eta readings visibly != 1/3 under twirled-native, and
    randomized-projective twirls readout-axis dephasing (T_zz = 1) to no
    noise at all.
    """
    print("\nTwo-protocol twirl: randomized-projective vs twirled-native "
          "estimator channels,\nexact per solid and draw (same probe noise "
          "as randomized_implementations.py):")
    # genericity: the two candidate factors differ, offset nonzero, T aniso
    assert abs(T_PROBE[2, 2] - np.trace(T_PROBE) / 3) > 0.05
    assert np.linalg.norm(t_PROBE) > 0.05
    assert np.linalg.norm(T_PROBE - np.trace(T_PROBE) / 3 * np.eye(3)) > 0.05

    rng = np.random.default_rng(7)           # local: irreducibility witness
    T_rand = rng.standard_normal((3, 3)) * 0.3
    t_rand = rng.standard_normal(3) * 0.2
    groups = {g: np.load(DATA / f"group_{g}.npz")["rotations"]
              for g in ("T", "O", "I")}
    r_test = np.array([0.31, -0.42, 0.53])   # fixed state inside the ball
    eta1 = T_PROBE[2, 2] / 3                 # R1 reading: T_zz / 3
    eta2 = np.trace(T_PROBE) / 9             # R2 reading: (tr T / 3) / 3

    # the cross-check the docstrings narrate, computed rather than asserted
    # in prose: the probe IS that module's, its two scalars are these two
    # after the factor 3, and its channels -- an independent expression that
    # carries the canonical dual's factor 3 -- are exactly 3x these.  Worst
    # deviation 1.1e-15 (icosahedron, both protocols).
    import randomized_implementations as ri  # local: import order untouched
    assert np.array_equal(T_PROBE, ri.T_NOISE), "probe T drifted apart"
    assert np.array_equal(t_PROBE, ri.t_NOISE), "probe t drifted apart"
    assert abs(3 * eta1 - ri.T_NOISE[2, 2]) < 1e-12           # R1: kappa
    assert abs(3 * eta2 - np.trace(ri.T_NOISE) / 3) < 1e-12   # R2: kappa
    s_x, R_x = povms["icosahedron"]["s"], groups["I"]
    for here, there in ((channel_R1, ri.channel_R1),
                        (channel_R2, ri.channel_R2)):
        M_h, m_h = here(s_x, R_x, T_PROBE, t_PROBE)
        M_t, m_t = there(s_x, R_x, ri.T_NOISE, ri.t_NOISE)
        assert np.abs(M_t - 3 * M_h).max() < 1e-12, here.__name__
        assert np.abs(m_t - 3 * m_h).max() < 1e-12, here.__name__
    print(f"  probe factors (3 eta): T_zz = {3 * eta1:.6f} "
          f"(randomized-projective) vs tr T/3 = {3 * eta2:.6f} "
          "(twirled-native)")

    out = {}
    rows = []
    for name in SOLIDS:
        s = povms[name]["s"]
        for draw, R in (("cov", groups[COVARIANCE_GROUP[name]]),
                        ("T", groups["T"])):
            M, m = channel_R2(s, R, T_PROBE, t_PROBE)
            aniso = np.abs(M - M[0, 0] * np.eye(3)).max()
            assert abs(M[0, 0] - eta2) < 1e-12 and aniso < 1e-12, (name, draw)
            assert np.abs(m).max() < 1e-12, (name, draw)
            assert np.abs((M @ r_test + m) / eta2 - r_test).max() < 1e-12
            Mr, mr = channel_R2(s, R, T_rand, t_rand)
            assert np.abs(Mr - (np.trace(T_rand) / 9)
                          * np.eye(3)).max() < 1e-12, (name, draw)
            assert np.abs(mr).max() < 1e-12, (name, draw)
            out[f"twirl2/{draw}/{name}"] = np.array(
                [M[0, 0], aniso, np.abs(m).max()])
        if name in ANTIPODAL:
            RG = groups[COVARIANCE_GROUP[name]]
            hits = orbit_hits(s, RG)         # the covariance draw realizes
            assert hits.min() == hits.max() == 2 * len(RG) // len(s), name
            M, m = channel_R1(s, RG, T_PROBE, t_PROBE)
            assert np.abs(M - eta1 * np.eye(3)).max() < 1e-12, name
            assert np.abs(m).max() < 1e-12, name
            assert np.abs((M @ r_test + m) / eta1 - r_test).max() < 1e-12
            Mr, mr = channel_R1(s, RG, T_rand, t_rand)
            assert np.abs(Mr - (T_rand[2, 2] / 3)
                          * np.eye(3)).max() < 1e-12, name
            assert np.abs(mr).max() < 1e-12, name
            out[f"twirl1/cov/{name}"] = np.array(
                [M[0, 0], np.abs(m).max(), hits[0]])
            # the minimal draw: T twirls every antipodal solid at the same
            # T_zz -- but its orbit realizes only octa/cube/icosa
            Mt, mt = channel_R1(s, groups["T"], T_PROBE, t_PROBE)
            assert np.abs(Mt - eta1 * np.eye(3)).max() < 1e-12, name
            assert np.abs(mt).max() < 1e-12, name
            th = orbit_hits(s, groups["T"])
            if name == "dodecahedron":       # 6/10 axes: realization fails
                assert sorted(set(th.tolist())) == [0, 2]
                assert int((th > 0).sum()) == 12
                torbit = "6/10 axes only -- NOT this POVM"
            else:
                assert th.min() == th.max() == 24 // len(s), name
                torbit = f"uniform x{th[0]}"
            out[f"twirl1/minT/{name}"] = np.array(
                [Mt[0, 0], th.min(), th.max(), int((th > 0).sum())])
            r1_cell = f"{M[0, 0]:.6f}"
        else:
            r1_cell, torbit = "-- (no antipodes)", "-- (R2 needs no orbit)"
        rows.append(f"  {name:14s}{r1_cell:>20s}{eta2:>12.6f}   {torbit}")
    print(f"  {'solid':14s}{'R1 eta = T_zz/3':>20s}{'R2 eta':>12s}   "
          "T-draw orbit")
    print("\n".join(rows))
    print("  [ok] R1 exactly depolarizing at T_zz/3 (antipodal four); R2 at "
          "(tr T/3)/3 for all\n  five (SIC included); offsets 0, inversion "
          "unbiased, both draws, to 1e-12")

    # the blindness study's channels, twirled: the blind spot closed exactly
    print("  blind-spot channels twirled (native |0> calibration reads 1/3 "
          "for every rate):")
    s_ico, s_tet = povms["icosahedron"]["s"], povms["tetrahedron"]["s"]
    zhat = np.array([0.0, 0.0, 1.0])
    for cname, chan in (("dephasing", chan_dephasing),
                        ("amp-damping", chan_amp_damping)):
        for p in BLIND_RATES:
            T3, t3 = chan(p)
            assert abs((T3 @ zhat + t3)[2] - 1.0) < 1e-12   # fixes |0>
            e2 = np.trace(T3) / 9
            for s_any in (s_ico, s_tet):     # solid-blind, SIC included
                M, m = channel_R2(s_any, groups["T"], T3, t3)
                assert np.abs(M - e2 * np.eye(3)).max() < 1e-12, (cname, p)
                assert np.abs(m).max() < 1e-12, (cname, p)
                assert np.abs((M @ r_test + m) / e2 - r_test).max() < 1e-12
            e1 = T3[2, 2] / 3
            M1, m1 = channel_R1(s_ico, groups["T"], T3, t3)
            assert np.abs(M1 - e1 * np.eye(3)).max() < 1e-12, (cname, p)
            assert np.abs(m1).max() < 1e-12, (cname, p)
            out[f"twirl2/blind/{cname}/{p}"] = np.array([M[0, 0]])
            out[f"twirl1/blind/{cname}/{p}"] = np.array([M1[0, 0]])
        readings = "  ".join(
            f"p={p}: {out[f'twirl2/blind/{cname}/{p}'][0]:.4f}"
            for p in BLIND_RATES)
        print(f"    {cname:12s} R2 eta {readings}")
    assert all(abs(out[f"twirl1/blind/dephasing/{p}"][0] - 1 / 3) < 1e-12
               for p in BLIND_RATES)         # T_zz = 1: twirled to no noise
    print("    (R1: dephasing has T_zz = 1 -- twirled to no noise at all; "
          "amp-damping reads (1-g)/3)")
    print("  [ok] each protocol's calibration sees the channels the native "
          "one cannot")
    return out


tp_out = two_protocol_twirl(povms)


Two-protocol twirl: randomized-projective vs twirled-native estimator channels,
exact per solid and draw (same probe noise as randomized_implementations.py):


  probe factors (3 eta): T_zz = 0.620000 (randomized-projective) vs tr T/3 = 0.720000 (twirled-native)
  solid              R1 eta = T_zz/3      R2 eta   T-draw orbit
  tetrahedron      -- (no antipodes)    0.240000   -- (R2 needs no orbit)
  octahedron                0.206667    0.240000   uniform x4
  cube                      0.206667    0.240000   uniform x3
  icosahedron               0.206667    0.240000   uniform x2
  dodecahedron              0.206667    0.240000   6/10 axes only -- NOT this POVM
  [ok] R1 exactly depolarizing at T_zz/3 (antipodal four); R2 at (tr T/3)/3 for all
  five (SIC included); offsets 0, inversion unbiased, both draws, to 1e-12
  blind-spot channels twirled (native |0> calibration reads 1/3 for every rate):
    dephasing    R2 eta p=0.01: 0.3289  p=0.05: 0.3111  p=0.1: 0.2889  p=0.2: 0.2444
    amp-damping  R2 eta p=0.01: 0.3311  p=0.05: 0.3222  p=0.1: 0.3108  p=0.2: 0.2877
    (R1: dephasing has T_zz = 1 -- twirled to no noise at all; amp-damping reads

### Reading the table

**The octahedron corner is the correlation story stripped bare.** Three separate facts conspire
there: $\hat z$ is already a vertex (so $A = \mathrm{Id}$), the native measurement is the uniform
mixture of the three Pauli bases (so no dilation), and the covariance group $O$ is exactly the
Bloch image of the single-qubit Clifford group $2O$. Both circuits collapse into "measure a
uniformly random Pauli" — standard randomized-Pauli shadows — *and the scalars still differ*:
$T_{zz} = 0.62$ against $\operatorname{tr}T/3 = 0.72$ on the shared probe. All that survives the
collapse is bookkeeping: whether the lab readout axis is fixed ($\hat z$, with the draw
re-orienting the state) or effectively ranges over the six vertices untied to the draw. Each
conspiring fact fails for every other solid: $\hat z$ is not a vertex of the cube, icosahedron or
dodecahedron (alignment nontrivial — and, by the field obstruction, *inexact*), and every other
solid's native circuit needs its dilation.

**The dodecahedron row splits the two jobs of the draw.** Under the minimal $T$-draw its channel
is *still exactly depolarizing* and the estimator unbiased — the twirl job never fails — but the
orbit covers only 6 of its 10 vertex axes: eight vertices are never sampled, so the draw
*realizes the wrong measurement*. Realization, not twirling, is what forces the dodecahedron up
to the full $2I$ draw (and its $0.8\,\Phi$ per shot, on the projective route alone).

**The blind spot does not survive either randomization.** Dephasing and amplitude damping — the
channels that pin the native calibration at $\frac13$ — twirl to $\eta$ readings the calibration
*can* see under twirled-native (e.g. $0.2889$ and $0.3108$ at rate $0.1$); and the projective
route twirls readout-axis dephasing ($T_{zz} = 1$) to *no noise at all*.

**Novelty ledger** (what is whose): the single-qubit core of the projective channel is Chen et
al.'s robust-shadow primitive — their uniformly-random Clifford + $Z$ readout *is* the
$2O$ protocol, their fidelity parameter is $\eta$. Ours: the two-channel distinction
($T_{zz}$ vs $\operatorname{tr}T/3$ — that the estimator-channel factor *identifies the protocol*),
the twirl that keeps the SIC, $2T$-universality with the realize/twirl split, and §9's gate-noise
pricing. Nguyen et al. (2022), the nearest neighbor, use Platonic transitivity to simplify the
canonical dual and model readout noise on the dilation ancillas — no group-averaging of noise
anywhere.

In [21]:
# === Demo: the five-liner mechanized, the two deaths, and a reducible counterexample ===

zhat = np.array([0.0, 0.0, 1.0])
R_I = np.load(DATA / "group_I.npz")["rotations"]
s_ico = povms["icosahedron"]["s"]
v = s_ico[np.argmax(s_ico[:, 2])]
A = align_to_z(v)

# R1's twirled object: the rank-one composition, conjugated into the solid's frame
X1 = A.T @ np.outer(zhat, zhat) @ T_PROBE @ A
tw1 = np.einsum("gji,jk,gkl->il", R_I, X1, R_I) / len(R_I)
assert np.abs(tw1 - (np.trace(X1) / 3) * np.eye(3)).max() < 1e-12
print(f"R1: twirl(zz^T T) = (tr/3) Id,  tr = T_zz = {np.trace(X1):.6f}")

# R2's twirled object: the noise alone
tw2 = np.einsum("gji,jk,gkl->il", R_I, T_PROBE, R_I) / len(R_I)
assert np.abs(tw2 - (np.trace(T_PROBE) / 3) * np.eye(3)).max() < 1e-12
print(f"R2: twirl(T)      = (tr/3) Id,  tr/3 = {np.trace(T_PROBE) / 3:.6f}")

# the offset's two deaths, separately
print(f"death 1 (solid):  max|sum of vertices| = "
      f"{np.abs(s_ico.sum(axis=0)).max():.1e}")
print(f"death 2 (group):  max|E_g R_g|         = "
      f"{np.abs(R_I.mean(axis=0)).max():.1e}")

# counterfactual: a REDUCIBLE twirl group -- the z-axis stabilizer inside O.
# Schur's step dies and the channel is no longer one scalar.
R_O = np.load(DATA / "group_O.npz")["rotations"]
stab = np.array([R for R in R_O if abs((R @ zhat)[2]) > 0.999])
print(f"\nz-axis stabilizer in O: {len(stab)} rotations "
      "(reducible -- it fixes the z line)")
twr = np.einsum("gji,jk,gkl->il", stab, T_PROBE, stab) / len(stab)
print("its twirl of the probe noise:")
print(np.round(twr, 6))
print("two scalars, not one: the x/y and z blocks average separately, so a "
      "single\ncalibrated eta cannot describe the channel -- the one-scalar "
      "claims of the robust\nprotocol die exactly at the lemma's "
      "'irreducibly'. (The dodecahedron's D5 story\nin "
      "randomized_implementations.py is this same failure inside I.)")

R1: twirl(zz^T T) = (tr/3) Id,  tr = T_zz = 0.620000
R2: twirl(T)      = (tr/3) Id,  tr/3 = 0.720000
death 1 (solid):  max|sum of vertices| = 0.0e+00
death 2 (group):  max|E_g R_g|         = 0.0e+00

z-axis stabilizer in O: 8 rotations (reducible -- it fixes the z line)
its twirl of the probe noise:
[[ 0.77  0.    0.  ]
 [-0.    0.77 -0.  ]
 [ 0.    0.    0.62]]
two scalars, not one: the x/y and z blocks average separately, so a single
calibrated eta cannot describe the channel -- the one-scalar claims of the robust
protocol die exactly at the lemma's 'irreducibly'. (The dodecahedron's D5 story
in randomized_implementations.py is this same failure inside I.)


## 9. Gate noise — the twirl's assumption, priced with the atlas

The twirl rests on one assumption: the noise must be **independent of the drawn rotation**. On
hardware each $g$ is its own circuit, so noise arrives per elementary gate, in an amount and
orientation *correlated with the draw* — and Schur's lemma is silent about a correlated average.
The assumption is not the same size for the two protocols: the projective route's whole circuit
varies with $g$; the twirled-native circuit varies only in its drawn $2T$ prefix — a Clifford
word of depth $\le 2$ — the dilation being the same circuit every shot.

**What the study computes.** One circuit per rotation from the atlas; amplitude damping of
strength $\gamma$ after every elementary gate of the drawn word; the estimator's channel averaged
*exactly* over the group; reported is the residual bias left on the TFIM site-0 Bloch components
after the $\ket{0}$-calibration — the error that survives the protocol's own correction, per unit
$\gamma$. The noise accounting per series:

- **projective rows** (`2O`, `2I`, `2I_phi3`, `2I_dij_phi3`, `2T`): damping on the drawn word
  only; the alignment is held ideal — a fixed gate's noise is $g$-independent and twirls cleanly,
  so charging it would only blur the correlated signal being measured. Because the noise *is*
  $g$-correlated, the residual depends on which vertex the alignment picks: the min/max over all
  vertex choices is stored alongside the convention's value.
- **`R2` rows**: damping on the same drawn $2T$ words; the fixed dilation is *modeled* as one
  $g$-independent pre-measurement amplitude-damping channel ($\Gamma = 0.05$) on the data qubit.
  A `R2bare` variant with no dilation channel isolates that modeling choice (it moves the
  calibration reading only; the residual shifts by under 2%).

**The two exact anchors** — the answer to "isn't the zero-residual claim circular, since you
modeled the dilation as removable?": at $\gamma = 0$ the residual is zero *against the visibly
noisy modeled dilation* — proving the twirl removes **whatever is $g$-independent**, which is a
theorem, not an assumption of convenience ($g$-independence of a fixed circuit is a statement
about the hardware layout, and the general theorem covers *any* such channel, ancilla-side
deformations included; only the location of the model is a choice, and `R2bare` prices it). And
gate-*independent* noise of any strength still twirls to an exact scalar with zero residual,
whatever the compilation. What the model is *not*: a hardware forecast. One species, one rate —
an illustration of the assumption's price, run on the atlas's own words.

In [22]:
# === Gate-noise study: per-gate damping on the atlas circuits ===

def bloch_rotation(U):
    """SO(3) rotation of conjugation by U: R_ij = tr(sigma_i U sigma_j U†)/2."""
    P = [PAULI["X"], PAULI["Y"], PAULI["Z"]]
    return np.array([[0.5 * np.trace(P[i] @ U @ P[j] @ U.conj().T).real
                      for j in range(3)] for i in range(3)])


def parse_sequence(seq):
    """Atlas sequence string -> [(gate, dagger), ...], operator order.

    Sequences are space-separated with 'I' for the identity; 'F X' means
    F.X, i.e. X is applied first (main.py's convention).  A trailing dagger
    marks the SU(2) inverse.
    """
    if seq == "I":
        return []
    return [(t.rstrip("†"), t.endswith("†")) for t in seq.split()]


def gate_noise_twirl(states, povms):
    """Per-gate noise breaks the exact twirl; the residual, quantified.

    twirl_check() assumes one noise channel independent of the drawn rotation
    g.  On hardware each g is a circuit from the atlas, so noise arrives per
    elementary gate and is *correlated with g* -- Schur's lemma no longer
    applies.  Per group element we compose the exact affine Bloch channel
    (T_g, t_g) of its synthesized circuit with amplitude damping gamma after
    every gate, then average the *estimator's* channel exactly -- readout
    probability times snapshot vertex, summed over (g, b), as in
    twirl_check():

        r -> M r + m,   M = E_g outer(R_g^T v, T_g^T v),
                        m = E_g (R_g^T v) (v . t_g),

    with v the vertex axis the fixed alignment identifies with the readout
    (v = z for the octahedron; the first vertex nearest +z for the
    icosahedron; the alignment itself is taken ideal -- its noise would be
    g-independent and twirl cleanly).  Reported: the anisotropy of M, the
    displacement |m|, the calibration reading eta = z.(M z + m) (ideal 1/3),
    and the exact residual bias (M r0 + m)/eta - r0 surviving the scalar
    |0> calibration.  Because the noise is g-correlated, the residual also
    depends on *which* vertex the alignment picks: the min/max of the Z0
    residual over all vertex choices is stored alongside the convention's
    value.  Series: min-depth (BFS) circuits for 2O and 2I at uniform
    gamma, plus 2I with Phi three times noisier -- magic cost read as noise
    cost -- compiled min-depth (BFS, up to two Phi per circuit) vs
    min-magic (Dijkstra, at most one); the "2T" series is the minimal
    projective draw -- the icosahedron realized and twirled by the uniform
    T draw (12 rotations, every word Clifford at depth <= 2, orbit = all
    12 vertices x2, same alignment vertex as the 2I series, so the
    residual drop is attributable to the draw alone); and the "R2" series
    prices the twirled-native protocol on the same T words.

    Noise accounting (a row is uninterpretable without it): every
    projective row (2O/2I/2T) damps every elementary gate of the DRAWN
    word only -- the alignment stays ideal as above.  The R2 rows damp
    every gate of the drawn T word and MODEL the fixed dilation as one
    g-independent pre-measurement amplitude-damping channel (GAMMA_DIL) on
    the data qubit: real dilation noise deforms the effects ancilla-side;
    the general twirl theorem covers any g-independent channel, and the
    pre-channel is the modeling choice.  It twirls out exactly -- at
    gamma = 0 the residual is zero against a visibly noisy dilation
    (eta = tr T_N/9, calibrated away) -- and the R2 estimator channel is
    provably solid-independent (the drawn words never see the solid), so
    the five per-solid rows are asserted equal to 1e-12: the SIC's noise
    bill equals the icosahedron's, and the whole g-correlated exposure
    sits in depth-<=2 Clifford words.  A bare variant (no dilation
    channel) isolates the modeling choice's effect.  Sanity limits:
    gamma = 0 recovers the ideal channel M = Id/3 with zero residual (for
    R2, M = (tr T_N/9) Id), and gate-independent noise of any strength
    still twirls to an exact scalar with zero residual, whatever the
    compilation -- both to machine precision.
    """
    print("\nGate-noise twirl study: per-gate amplitude damping on the "
          "atlas circuits\n(noise correlated with g -- the exact twirl "
          "assumption dropped):")
    gates = np.load(DATA / "gates.npz")
    su2 = {("Φ" if str(n) == "Phi" else str(n)): U
           for n, U in zip(gates["names"], gates["su2"])}
    rot = {n: bloch_rotation(U) for n, U in su2.items()}
    zhat = np.array([0.0, 0.0, 1.0])
    r0 = site_bloch(states["TFIM"])[0]     # the blindness study's test vector

    def circuit_channel(tokens, gamma, phi_mult):
        """Affine Bloch channel of the circuit, noise after every gate."""
        T, t = np.eye(3), np.zeros(3)
        for base, dag in reversed(tokens):      # rightmost gate acts first
            R = rot[base].T if dag else rot[base]
            T, t = R @ T, R @ t
            g = gamma * (phi_mult if base == "Φ" else 1.0)
            if g > 0.0:
                Tn, tn = chan_amp_damping(g)
                T, t = Tn @ T, Tn @ t + tn
        return T, t

    def load_series(gname, mode):
        """One circuit per SO(3) rotation: min depth (bfs) or min (magic,
        depth) (dij) within each +/-q pair; replay-checked against the
        stored unitaries."""
        d = np.load(DATA / f"group_{gname}.npz")
        Us, seqs = d["unitaries"], d[f"{mode}_sequences"]
        score = d["bfs_depths"] if mode == "bfs" else \
            list(zip(d["dij_magic_costs"], d["dij_depths"]))
        reps = {}
        for i in range(len(Us)):
            key = tuple(np.round(bloch_rotation(Us[i]), 9).ravel())
            if key not in reps or score[i] < score[reps[key]]:
                reps[key] = i
        idx = sorted(reps.values())
        assert len(idx) == len(Us) // 2, gname
        Rs, toks, phis = [], [], []
        for i in idx:
            tk = parse_sequence(seqs[i])
            U = I2.copy()
            for base, dag in tk:               # operator order replay
                U = U @ (su2[base].conj().T if dag else su2[base])
            assert np.abs(U - Us[i]).max() < 1e-10, (gname, mode, i)
            Rs.append(bloch_rotation(Us[i]))
            toks.append(tk)
            phis.append(sum(1 for b, _ in tk if b == "Φ"))
        return np.stack(Rs), toks, np.array(phis)

    def estimator_channel(Rs, chans, v):
        """(M, m) of the randomized estimator for alignment vertex v."""
        M = np.mean([np.outer(R.T @ v, T.T @ v)
                     for R, (T, _) in zip(Rs, chans)], axis=0)
        m = np.mean([(R.T @ v) * (v @ t)
                     for R, (_, t) in zip(Rs, chans)], axis=0)
        return M, m

    def residual(M, m):
        """Post-|0>-calibration bias on r0, and the calibration reading."""
        eta = (M @ zhat + m)[2]
        return (M @ r0 + m) / eta - r0, eta

    series = {"2O": ("2O", "bfs", 1.0), "2I": ("2I", "bfs", 1.0),
              "2I_phi3": ("2I", "bfs", 3.0),
              "2I_dij_phi3": ("2I", "dij", 3.0),
              "2T": ("2T", "bfs", 1.0)}
    verts = {"2O": povms["octahedron"]["s"], "2I": povms["icosahedron"]["s"],
             "2T": povms["icosahedron"]["s"]}
    stash = {}                                 # the T words, reused by R2
    out = {}
    for name, (gname, mode, phi_mult) in series.items():
        Rs, toks, phis = load_series(gname, mode)
        depths = np.array([len(tk) for tk in toks])
        if mode == "dij":                      # the magic-cost dichotomy
            assert phis.max() <= 1, name
        out[f"gatenoise/meta/{name}"] = np.array(
            [len(Rs), depths.mean(), depths.max(), phis.mean()])
        s = verts[gname]
        v = s[np.argmax(s[:, 2])]              # alignment convention
        if gname == "2O":
            assert np.abs(v - zhat).max() < 1e-12   # z already a vertex
        if name == "2T":                       # the minimal projective draw
            assert depths.max() <= 2 and int(phis.max()) == 0, name
            assert (orbit_hits(s, Rs) == 2).all()   # all 12 vertices, x2
            stash["T"] = (Rs, toks, depths, phis)

        # sanity: gate-independent noise after the whole (ideal) circuit
        # still twirls to an exact scalar with zero residual (Schur).  The
        # compilation cannot enter: the damping lands after the whole ideal
        # circuit, so `toks` is never read below and the three icosahedral
        # rows store bitwise-identical values.  The control row's "every
        # compilation" is therefore true by construction, not by this sweep.
        # Stored as well as asserted: Appendix F's table prints the zero as a
        # literal, so what keeps the page honest is shadow_report.py
        # re-asserting the stored value before it typesets -- a guard that
        # sits downstream of this file, and so survives THIS file's tolerance
        # being loosened.  The row claims ANY strength, so the check sweeps
        # strengths rather than pinning one and leaving Schur to cover the
        # difference.
        worst = np.zeros(2)
        for delta in INDEP_DELTAS:
            Tn, tn = chan_amp_damping(delta)
            M0, m0 = estimator_channel(Rs, [(Tn @ R, tn) for R in Rs], v)
            assert np.abs(M0 - (v @ Tn @ v / 3) * np.eye(3)).max() < 1e-12
            assert np.abs(m0).max() < 1e-12
            bias0 = residual(M0, m0)[0]
            assert np.abs(bias0).max() < 1e-12, (name, delta)
            worst = np.maximum(worst, np.abs(bias0[[0, 2]]))
        out[f"gatenoise/indep/{name}"] = worst    # worst |(X0, Z0)| over the sweep

        for gamma in GATE_GAMMAS:
            chans = [circuit_channel(tk, gamma, phi_mult) for tk in toks]
            M, m = estimator_channel(Rs, chans, v)
            aniso = np.abs(M - (np.trace(M) / 3) * np.eye(3)).max()
            disp = np.linalg.norm(m)
            bias, eta = residual(M, m)
            out[f"gatenoise/{name}/{gamma}"] = np.array(
                [aniso, disp, eta, bias[0], bias[2]])
            span = [residual(*estimator_channel(Rs, chans, vk))[0][2]
                    for vk in s]
            out[f"gatenoise/span/{name}/{gamma}"] = np.array(
                [min(span), max(span)])
        g0 = out[f"gatenoise/{name}/0.0"]
        assert np.abs(g0[[0, 1, 3, 4]]).max() < 1e-12
        assert abs(g0[2] - 1 / 3) < 1e-12
        print(f"  {name:12s} rotations {len(Rs):3d}, mean depth "
              f"{depths.mean():.2f}, mean Phi {phis.mean():.2f}: "
              "gamma=0 -> ideal, gate-independent -> Schur: OK")

    # ---- the twirled-native series: the same per-gate damping on the T
    # words; the fixed dilation MODELED as one g-independent pre-measurement
    # channel (amplitude damping, GAMMA_DIL) on the data qubit, which the
    # twirl removes exactly -- so what remains is the drawn words' bill,
    # and the estimator channel is provably solid-independent.
    Rs, toks, depths, phis = stash["T"]
    TN, tN = chan_amp_damping(GAMMA_DIL)
    eta_dil = np.trace(TN) / 9               # the gamma = 0 reading
    out["gatenoise/meta/R2"] = np.array(
        [len(Rs), depths.mean(), depths.max(), phis.mean()])
    for gamma in GATE_GAMMAS:
        chans = [circuit_channel(tk, gamma, 1.0) for tk in toks]
        M_cf = np.mean([R.T @ TN @ T                 # closed form: Id/3
                        for R, (T, _) in zip(Rs, chans)], axis=0) / 3.0
        ref = None
        for sname in SOLIDS:
            sv = povms[sname]["s"]
            C = sv.T @ sv / len(sv)          # the solid enters here only --
            M = np.mean([R.T @ C @ TN @ T    # -- and contracts to Id/3
                         for R, (T, _) in zip(Rs, chans)], axis=0)
            m = np.mean([R.T @ (sv.mean(axis=0) + C @ (TN @ t + tN))
                         for R, (_, t) in zip(Rs, chans)], axis=0)
            assert np.abs(M - M_cf).max() < 1e-12, (sname, gamma)
            aniso = np.abs(M - (np.trace(M) / 3) * np.eye(3)).max()
            bias, eta = residual(M, m)
            row = np.array([aniso, np.linalg.norm(m), eta, bias[0], bias[2]])
            out[f"gatenoise/R2/{sname}/{gamma}"] = row
            if ref is None:
                ref = row
            assert np.abs(row - ref).max() < 1e-12, (sname, gamma)
        # bare variant: no dilation channel -- the modeling choice isolated
        Mb = np.mean([R.T @ T for R, (T, _) in zip(Rs, chans)], axis=0) / 3.0
        mb = np.mean([R.T @ t for R, (_, t) in zip(Rs, chans)], axis=0) / 3.0
        biasb, etab = residual(Mb, mb)
        out[f"gatenoise/R2bare/{gamma}"] = np.array(
            [np.abs(Mb - (np.trace(Mb) / 3) * np.eye(3)).max(),
             np.linalg.norm(mb), etab, biasb[0], biasb[2]])
    # the twirled-native counterpart of the projective control above, and what
    # lets Appendix F's second control row say "both protocols" rather than
    # "the projective series": at gamma = 0 the drawn words are ideal and the
    # only noise present is the dilation, which is g-independent by
    # construction -- so sweeping ITS strength is the same test.
    sv = povms["icosahedron"]["s"]
    C = sv.T @ sv / len(sv)
    worst = np.zeros(2)
    for delta in INDEP_DELTAS:
        Td, td = chan_amp_damping(delta)
        M = np.mean([R.T @ C @ Td @ R for R in Rs], axis=0)
        m = np.mean([R.T @ (sv.mean(axis=0) + C @ td) for R in Rs], axis=0)
        bias0 = residual(M, m)[0]
        assert np.abs(bias0).max() < 1e-12, delta
        worst = np.maximum(worst, np.abs(bias0[[0, 2]]))
    out["gatenoise/indep/R2"] = worst
    g0 = out["gatenoise/R2/icosahedron/0.0"]
    assert np.abs(g0[[0, 1, 3, 4]]).max() < 1e-12    # the exact anchor:
    assert abs(g0[2] - eta_dil) < 1e-12              # dilation twirls out
    print(f"  {'R2':12s} rotations {len(Rs):3d}, mean depth "
          f"{depths.mean():.2f}, mean Phi {phis.mean():.2f}: gamma=0 -> "
          f"zero residual against the\n{'':15s}dilation channel "
          f"(eta = tr T_N/9 = {eta_dil:.4f}, calibrated away): OK; "
          "channel\n" + " " * 15 + "solid-independent (all five rows equal "
          "to 1e-12, SIC included): OK")

    print("    series        " + "".join(f"{f'g={g}':>11s}"
                                         for g in GATE_GAMMAS[1:]))
    for name in series:
        row = [out[f"gatenoise/{name}/{g}"][4] for g in GATE_GAMMAS[1:]]
        print(f"    {name:12s}  " + "".join(f"{b:+11.5f}" for b in row)
              + "   (residual Z0 bias)")
    row = [out[f"gatenoise/R2/icosahedron/{g}"][4] for g in GATE_GAMMAS[1:]]
    print(f"    {'R2 (any)':12s}  " + "".join(f"{b:+11.5f}" for b in row)
          + "   (residual Z0 bias)")
    g1 = GATE_GAMMAS[1]
    print(f"    alignment span of the Z0 residual at g={g1} "
          "(min .. max over vertex choices; R2 has no alignment):")
    for name in series:
        lo, hi = out[f"gatenoise/span/{name}/{g1}"]
        print(f"    {name:12s}  {lo:+.6f} .. {hi:+.6f}")
    return out


gate_out = gate_noise_twirl(states, povms)


Gate-noise twirl study: per-gate amplitude damping on the atlas circuits
(noise correlated with g -- the exact twirl assumption dropped):
  2O           rotations  24, mean depth 1.71, mean Phi 0.00: gamma=0 -> ideal, gate-independent -> Schur: OK
  2I           rotations  60, mean depth 2.43, mean Phi 0.92: gamma=0 -> ideal, gate-independent -> Schur: OK
  2I_phi3      rotations  60, mean depth 2.43, mean Phi 0.92: gamma=0 -> ideal, gate-independent -> Schur: OK
  2I_dij_phi3  rotations  60, mean depth 2.53, mean Phi 0.80: gamma=0 -> ideal, gate-independent -> Schur: OK
  2T           rotations  12, mean depth 1.50, mean Phi 0.00: gamma=0 -> ideal, gate-independent -> Schur: OK
  R2           rotations  12, mean depth 1.50, mean Phi 0.00: gamma=0 -> zero residual against the
               dilation channel (eta = tr T_N/9 = 0.3222, calibrated away): OK; channel
               solid-independent (all five rows equal to 1e-12, SIC included): OK
    series            g=0.001    g=0.002  

### Reading the residuals

Every projective series is **linear in $\gamma$**, and the slope tracks how the noise correlates
with the draw — not any single cost column: $2I$ (mean depth 2.43, $0.92\,\Phi$ per rotation)
pays about twice $2O$ (depth 1.71, Clifford); tripling the damping on $\Phi$ alone — magic cost
read as noise cost — more than doubles $2I$'s bill; and recompiling min-magic (Dijkstra)
*raises* the $Z_0$ residual by about a third while carrying **less** $\Phi$ (0.80 vs 0.92) — a
cheaper magic bill is not a cheaper correlation bill. Swapping the icosahedron's full-$2I$ draw
for the minimal $2T$ one — same POVM, same alignment vertex, twelve all-Clifford words of depth
$\le 2$ — roughly **halves** the correlated exposure (and flips the sign: that is a property of
the convention vertex, as the alignment-span rows show; the magnitude is not).

The `R2` row is the headline: its $Z_0$ residual is **second order** ($-0.082\,\gamma^2$ — four
orders below the projective rows at $\gamma = 10^{-3}$), its $X_0$ slope about half the projective
$2T$ row's, and the channel is **provably solid-independent** — the drawn words never see which
POVM the dilation implements, so the five per-solid rows are asserted equal to $10^{-12}$ and one
bill prices every solid. For the tetrahedral SIC, which no projective protocol admits, these are
its first noise numbers in the thesis.

That second order is not luck, and the next two cells prove why.

### Cracking the second order

Expand the noisy word channel to first order in $\gamma$. Amplitude damping after one gate
contributes the generator pair $D = \mathrm{diag}(\frac12, \frac12, 1)$ (matrix decay) and
$\hat z$ (displacement), inserted at that point of the word: writing the word's rotation as
$R_g = S\,P$ — suffix times prefix at the insertion — the first-order channel error is a sum of
insertions $S\,D\,P$ and displacements $S\,\hat z$. Three structural facts finish it:

1. **The estimator sees each insertion through its prefix alone.** Twirled-native post-processing
   applies $R_g^\top$ to the snapshot, and orthogonality cancels the suffix:
   $R_g^\top (S\,D\,P) = P^\top D\,P$ (with the modeled dilation in front,
   $R_g^\top T_N S\,D\,P = P^\top (S^\top T_N S)\,D\,P$ — the suffix survives only inside a
   diagonal sandwich). The first-order average is a sum over the **prefix multiset** of the twelve
   drawn words.
2. **Nothing first-order can tilt axes.** Every rotation in $T$ is a *signed coordinate
   permutation*, and $D$, $T_N$ are diagonal — so every term above is diagonal: at order $\gamma$
   the twirled channel error can **rescale axes and displace along axes, but never mix them**.
   ($O$ is also all signed permutations, so this holds for the octahedral projective row too —
   diagonality alone is not the protocol difference.)
3. **A scalar $z$-calibration absorbs a diagonal $z$-error exactly.** Writing the first-order
   channel as $M \approx \eta_0\mathrm{Id} + \gamma M_1$, $m \approx \gamma m_1$ with $M_1$
   diagonal, the calibrated residual's linear $Z_0$ coefficient collapses to
   $$\frac{d}{d\gamma}\Bigl[\text{residual}_z\Bigr]_{\gamma=0} =
   \frac{(m_1)_z\,(1 - z_0)}{\eta_0}$$
   — the matrix part cancels *identically* between numerator and calibration. Only an axis-aligned
   **$z$-displacement** could survive at first order — and for the twelve atlas words the $\pm$
   prefix contributions along $z$ cancel **exactly**: $(m_1)_z = 0$, verified by direct count
   below. So the $Z_0$ residual starts at $\gamma^2$.

What stays linear, and why, completes the picture. The *transverse* displacement does not cancel
($m_1 \propto (1,1,0)$), and no scalar calibration exists to absorb it: the $X_0$ slope is
$(m_1)_x/\eta_0$ — for the bare variant exactly $\frac{1/18}{1/3} = \frac16$ — plus a whisker of
$\Gamma$-induced diagonal anisotropy. And the **projective** route stays linear in $Z_0$ because
its $g$-correlated snapshot *weight* deposits a first-order $z$-displacement: for the octahedral
draw — where step 2's diagonality argument applies just as well — the entire $+0.250$ slope is
displacement, $(m_1)_z = \frac1{12}$ exactly, and $3 \times \frac1{12} = 0.25$. The protocol
difference at first order is *where the displacement goes*: the projective correlation pushes it
onto the readout axis; the twirled-native relabeling average cancels it there.

In [23]:
# === Demo: the first-order error, assembled by hand from the word prefixes ===

gates_npz = np.load(DATA / "gates.npz")
su2 = {("Φ" if str(n) == "Phi" else str(n)): U
       for n, U in zip(gates_npz["names"], gates_npz["su2"])}
rot = {n: bloch_rotation(U) for n, U in su2.items()}
d2t = np.load(DATA / "group_2T.npz")
Us, seqs, score = d2t["unitaries"], d2t["bfs_sequences"], d2t["bfs_depths"]
reps = {}
for i in range(len(Us)):        # one circuit per rotation, as in the study
    key = tuple(np.round(bloch_rotation(Us[i]), 9).ravel())
    if key not in reps or score[i] < score[reps[key]]:
        reps[key] = i
idx = sorted(reps.values())
words = [parse_sequence(seqs[i]) for i in idx]
Rlist = [bloch_rotation(Us[i]) for i in idx]
print("the twelve drawn words:",
      [" ".join(b + ("†" if dg else "") for b, dg in tk) or "I"
       for tk in words])

# fact: every rotation in T is a signed coordinate permutation
assert all(set(np.unique(np.abs(np.round(R, 12)))) <= {0.0, 1.0}
           for R in Rlist)

zhat = np.array([0.0, 0.0, 1.0])
D_gen = np.diag([0.5, 0.5, 1.0])            # amplitude damping: -dT/dgamma
TN, tN = chan_amp_damping(GAMMA_DIL)        # the modeled dilation channel
M1 = np.zeros((3, 3)); m1 = np.zeros(3)     # first-order error, dilated
m1_bare = np.zeros(3)                       # and the bare displacement
for tk, Rg in zip(words, Rlist):
    P = np.eye(3)
    for base, dag in reversed(tk):          # gates in time order
        P = (rot[base].T if dag else rot[base]) @ P
        S = Rg @ P.T                        # suffix: S P = R_g
        mid = S.T @ TN @ S                  # diagonal (signed permutations)
        M1 -= P.T @ mid @ D_gen @ P / (3 * 12)
        m1 += P.T @ mid @ zhat / (3 * 12)
        m1_bare += P.T @ zhat / (3 * 12)

off = np.abs(M1 - np.diag(np.diag(M1))).max()
print(f"\nM1 off-diagonal = {off}  (exactly diagonal)")
print(f"diag(M1) = {np.round(np.diag(M1), 6)}")
print(f"m1 = {np.round(m1, 6)}   (m1)_z = {m1[2]}  (exact cancellation)")
assert off == 0.0 and m1[2] == 0.0

# the scalar calibration absorbs the diagonal z-error: linear slopes
r0 = site_bloch(states["TFIM"])[0]
eta0 = np.trace(TN) / 9
eta1 = M1[2, 2] + m1[2]
lin = (M1 @ r0 + m1 - r0 * eta1) / eta0
g = GATE_GAMMAS[1]
row = gate_out[f"gatenoise/R2/icosahedron/{g}"]
print(f"\npredicted linear slopes:  X0 = {lin[0]:+.6f}   Z0 = {lin[2]:+.6f}")
print(f"study, at gamma={g}:      X0/g = {row[3] / g:+.6f}   "
      f"Z0/g = {row[4] / g:+.6f}   Z0/g^2 = {row[4] / g ** 2:+.4f}")
assert lin[2] == 0.0 and abs(lin[0] - row[3] / g) < 1e-3
print(f"bare closed form: X0 slope = 3*(m1_bare)_x = {3 * m1_bare[0]:.6f} "
      f"= 1/6;  study: {gate_out[f'gatenoise/R2bare/{g}'][3] / g:+.6f}")

# contrast: the projective route on the SAME words -- the snapshot weight is
# g-correlated, and its first-order z-displacement is what stays linear
def r1_channel(gamma):
    M = np.zeros((3, 3)); m = np.zeros(3)
    v = s_ico[np.argmax(s_ico[:, 2])]
    for tk, Rg in zip(words, Rlist):
        T, t = np.eye(3), np.zeros(3)
        for base, dag in reversed(tk):
            R = rot[base].T if dag else rot[base]
            T, t = R @ T, R @ t
            if gamma > 0:
                Tn, tn = chan_amp_damping(gamma)
                T, t = Tn @ T, Tn @ t + tn
        M += np.outer(Rg.T @ v, T.T @ v) / 12
        m += (Rg.T @ v) * (v @ t) / 12
    return M, m

eps = 1e-7
m1_proj = (r1_channel(eps)[1] - r1_channel(0.0)[1]) / eps
print(f"\nprojective 2T draw (icosahedron vertex): first-order z-displacement "
      f"= {m1_proj[2]:+.6f}")
print(f"study's 2T Z0 slope: {gate_out[f'gatenoise/2T/{g}'][4] / g:+.6f}   "
      "(nonzero displacement -> linear; the scalar cannot absorb it)")

the twelve drawn words: ['I', 'F', 'F†', 'X', 'Z', 'F X', 'F† Z', 'X F', 'X F†', 'X Z', 'Z F', 'Z F†']

M1 off-diagonal = 0.0  (exactly diagonal)
diag(M1) = [-0.320437 -0.320437 -0.322151]
m1 = [0.052778 0.052778 0.      ]   (m1)_z = 0.0  (exact cancellation)

predicted linear slopes:  X0 = +0.167305   Z0 = +0.000000
study, at gamma=0.001:      X0/g = +0.167387   Z0/g = -0.000082   Z0/g^2 = -0.0820
bare closed form: X0 slope = 3*(m1_bare)_x = 0.166667 = 1/6;  study: +0.166748

projective 2T draw (icosahedron vertex): first-order z-displacement = -0.097568
study's 2T Z0 slope: -0.292988   (nonzero displacement -> linear; the scalar cannot absorb it)


## 10. $n$-scaling — nothing above is an $n = 4$ artifact

**The job:** rule out that Experiment 2's gains evaporate — or explode — as the chain grows.

**Why this is computable exactly at $n = 16$**, where the outcome tensor would have $12^{16}$
entries: the variance is a sum over *term pairs*, and each pair's expectation touches only the
reduced state on the union of the two supports — at most 4 sites for the TFIM. Sparse-Lanczos
ground states plus $\le 4$-site reduced density matrices make every dual level's single-shot
variance an exact number at any $n$ we care to reach (the $n = 4$ values are asserted equal to
the full outcome-tensor route).

**The dual-independent floor.** For a support-*disjoint* pair the per-site operators reduce to
plain Paulis — the middle case of the site-cases identity, i.e. the frame condition, valid for
*every* dual — so disjoint-pair covariances are identical for all duals, and optimization can
only touch the shared-site block. That floor is exactly the long-range-correlation share one
might fear would dominate at criticality; it never exceeds $1.7\%$ of the variance at any size
computed.

**The verdict:** the per-site variance is flat within two percent ($12.41 \to 12.18$), and the
optimization gain **converges, not collapses** — oracle $15.6\% \to 15.1\%$, observable-optimized
$\to 14.1\%$. The $n = 4$ tables were representative all along; what caps the gain is the
per-site quantum limit of the *factorized estimator class*, not anything the chain grows into.

And the honest defense of that mildly negative headline — "fifteen percent, so what?": the
ceiling is a *structural* fact about the scalable class (leaving it — joint or tensor-network
duals — buys orders of magnitude, at exponential or network cost), and knowing the exact
accounting changes what a deployer does: pick the cube for tails (§4), expect no ranking at the
canonical dual (§3), budget the calibration premium (§6), keep $\hat v_\perp$ (§7), calibrate
the scalar *of the protocol actually run* (§8), and prefer the twirled-native route when gate
noise dominates (§9).

In [24]:
# === n-scaling: exact variance out to n = 16 ===

def tfim_terms_n(n, g):
    """Periodic TFIM terms at arbitrary size (tfim_terms is N_QUBITS-bound)."""
    terms = [(-1.0, [(i, "Z"), ((i + 1) % n, "Z")]) for i in range(n)]
    terms += [(-g, [(i, "X")]) for i in range(n)]
    return terms


def tfim_ground_sparse(n, g):
    """Ground energy and state of the periodic TFIM, sparse Lanczos."""
    dim = 1 << n
    basis = np.arange(dim)
    diag = np.zeros(dim)
    for i in range(n):                     # -sum Z_i Z_{i+1}: diagonal
        zi = 1 - 2 * ((basis >> (n - 1 - i)) & 1)
        zj = 1 - 2 * ((basis >> (n - 1 - (i + 1) % n)) & 1)
        diag -= zi * zj
    rows, cols, vals = [basis], [basis], [diag]
    for i in range(n):                     # -g sum X_i: bit flips
        rows.append(basis)
        cols.append(basis ^ (1 << (n - 1 - i)))
        vals.append(np.full(dim, -g))
    H = coo_matrix((np.concatenate(vals),
                    (np.concatenate(rows), np.concatenate(cols))),
                   shape=(dim, dim)).tocsr()
    v0 = np.full(dim, dim ** -0.5)         # deterministic start (H stoquastic
    w, v = eigsh(H, k=1, which="SA", v0=v0)  # -> overlap with ground state)
    return w[0], v[:, 0]


def tfim_ground_energy_exact(n, g):
    """Closed-form ground energy of the periodic TFIM, by Jordan-Wigner.

    H = -sum_i Z_i Z_{i+1} - g sum_i X_i on a ring of n sites is free fermions.
    For even n the ground state sits in the even-parity (Neveu-Schwarz) sector,
    whose momenta are the half-integers k = (2m+1) pi / n, and

        E_0 = -(1/2) sum_k eps_k,   eps_k = 2 sqrt(1 + g^2 - 2 g cos k).

    This exists to be an INDEPENDENT VERIFIER of the two ground states this
    file builds, and it shares no mechanism with either: it constructs no
    Hamiltonian, diagonalizes nothing, and samples nothing.  Both routes are
    checked against it -- the dense eigh at n = 4, whose state anchors every
    number in Appendix F and whose energy the thesis quotes as -5.23, and the
    sparse Lanczos across n = 4..16 behind the n-scaling figure.
    """
    ks = (np.pi * (2 * m + 1) / n for m in range(n))
    return -0.5 * sum(2 * np.sqrt(1 + g ** 2 - 2 * g * np.cos(k)) for k in ks)


def reduced_rho(psi, n, sites):
    """Reduced density matrix of |psi> on the given (sorted) sites."""
    rest = [i for i in range(n) if i not in sites]
    M = psi.reshape([2] * n).transpose(list(sites) + rest)
    M = M.reshape(2 ** len(sites), -1)
    return M @ M.conj().T


def site_pair_ops(s, x):
    """Per-site 2x2 operators of the factorized estimator.

    x has shape (3, V): x[a, k] is the per-shot value for Pauli letter a on
    outcome k.  single[a] = sum_k x[a,k] E_k is what a site one term touches
    contributes -- the frame condition makes it exactly sigma_a, for *every*
    valid dual (asserted).  pair[a, b] = sum_k x[a,k] x[b,k] E_k is the
    dual-dependent operator for a site both terms touch.
    """
    V = len(s)
    E = np.stack([(I2 + s[k, 0] * PAULI["X"] + s[k, 1] * PAULI["Y"]
                   + s[k, 2] * PAULI["Z"]) / V for k in range(V)])
    single = np.einsum("ak,kij->aij", x, E)
    pair = np.einsum("ak,bk,kij->abij", x, x, E)
    for a, name in enumerate("XYZ"):       # frame condition, exactly
        assert np.abs(single[a] - PAULI[name]).max() < 1e-11
    return single, pair


def exact_energy_variance(psi, n, terms, s, x, rho_cache):
    """Exact single-shot variance of the factorized energy estimator.

    Var = sum_{t,t'} c_t c_t' (E[o_t o_t'] - <P_t><P_t'>), with each
    E[o_t o_t'] = Tr[rho_S (x) per-site ops] on the <= 4-site union S of the
    two supports.  Support-disjoint pairs reduce to the plain quantum
    covariance <P_t P_t'> - <P_t><P_t'> for every valid dual (the frame
    condition again; asserted en route), so that part of the variance -- the
    'floor', returned alongside -- is dual-independent: optimization can only
    touch the overlapping block.
    """
    single, pair = site_pair_ops(s, x)
    sigma = {a: PAULI[nm] for a, nm in enumerate("XYZ")}

    def rho_S(S):
        if S not in rho_cache:
            rho_cache[S] = reduced_rho(psi, n, list(S))
        return rho_cache[S]

    supports = [dict((i, AXIS[a]) for i, a in sites) for _, sites in terms]
    coeffs = np.array([c for c, _ in terms])
    means = np.empty(len(terms))
    for ti, sup in enumerate(supports):
        S = tuple(sorted(sup))
        means[ti] = np.trace(rho_S(S) @ kron_all(
            [sigma[sup[i]] for i in S])).real
    e_mean = coeffs @ means

    second, floor, worst = 0.0, 0.0, 0.0
    for ti, s1 in enumerate(supports):
        for tj, s2 in enumerate(supports):
            S = tuple(sorted(set(s1) | set(s2)))
            ops = [pair[s1[i], s2[i]] if (i in s1 and i in s2)
                   else sigma[s1[i]] if i in s1 else sigma[s2[i]]
                   for i in S]
            val = np.trace(rho_S(S) @ kron_all(ops)).real
            second += coeffs[ti] * coeffs[tj] * val
            if not set(s1) & set(s2):      # disjoint: dual-independent
                pval = np.trace(rho_S(S) @ kron_all(
                    [sigma[(s1 | s2)[i]] for i in S])).real
                worst = max(worst, abs(val - pval))
                floor += coeffs[ti] * coeffs[tj] * (pval - means[ti] * means[tj])
    assert worst < 1e-10
    return second - e_mean ** 2, floor, e_mean


def nscaling(povms, states):
    """Exact variance of the factorized energy estimator, n = 4 .. 16.

    Resolves the 'everything is at n = 4' caveat by computing the icosahedral
    estimator's exact single-shot variance on the critical TFIM energy per
    dual level (canonical / observable-optimized / per-site oracle) as the
    chain grows.  The per-site variance and the optimization gain both
    converge fast -- the gain to a constant ~15%, not to zero -- and the
    dual-independent covariance floor stays a small share, so what caps the
    gain is the per-site quantum limit of the factorized class, not
    long-range correlations.  Cross-checks: at n = 4 the marginal route
    equals the full outcome-tensor route on all three levels, and the
    canonical-dual curve is identical for octahedron and icosahedron
    (the exact-landscape proposition, at every n).
    """
    print("\nn-scaling: exact variance of the factorized TFIM-energy "
          f"estimator, icosahedron,\nn = {NSCALE_NS[0]}..{NSCALE_NS[-1]} "
          "(sparse ground states; canonical / observable-opt / oracle):")
    s_ico = povms["icosahedron"]["s"]
    s_oct = povms["octahedron"]["s"]

    def x_canonical(s):
        return 3.0 * s.T.copy()

    def x_letter(s, rbar):
        b = letter_duals(s, rbar)
        return np.stack([2.0 * b[a][:, a] for a in range(3)])

    out = {}
    print(f"    {'n':>2s} {'E0':>10s} {'<X>':>8s} {'Var can':>10s} "
          f"{'Var obs':>10s} {'Var orc':>10s} {'floor':>8s} {'gain':>7s}")
    for n in NSCALE_NS:
        terms = tfim_terms_n(n, G_CRIT)
        e0, psi = tfim_ground_sparse(n, G_CRIT)
        # Lanczos converged on the ground state, not on a low excited one --
        # the free-fermion energy is an independent witness at every n.
        assert abs(e0 - tfim_ground_energy_exact(n, G_CRIT)) < 1e-9, (n, e0)
        cache = {}
        xs = []
        for g in TRAIN_G:                  # training mean, as in Experiment 2
            _, pg = tfim_ground_sparse(n, g)
            r1 = reduced_rho(pg, n, [0])
            xs.append([np.trace(r1 @ PAULI[a]).real for a in "XYZ"])
        rbar = np.mean(xs + [[0.0, 0.0, 0.0]], axis=0)      # + GHZ (= 0)
        r1 = reduced_rho(psi, n, [0])
        r_true = np.array([np.trace(r1 @ PAULI[a]).real for a in "XYZ"])

        var_can, floor, e_mean = exact_energy_variance(
            psi, n, terms, s_ico, x_canonical(s_ico), cache)
        var_obs, floor2, _ = exact_energy_variance(
            psi, n, terms, s_ico, x_letter(s_ico, rbar), cache)
        var_orc, floor3, _ = exact_energy_variance(
            psi, n, terms, s_ico, x_letter(s_ico, r_true), cache)
        assert abs(floor - floor2) < 1e-9 and abs(floor - floor3) < 1e-9
        assert abs(e_mean - e0) < 1e-8
        var_oct, _, _ = exact_energy_variance(
            psi, n, terms, s_oct, x_canonical(s_oct), cache)
        assert abs(var_oct - var_can) < 1e-9       # antipodal universality

        if n == 4:                         # cross-route + cross-module checks
            rho4 = states["TFIM"]
            assert abs(e0 - exact_value(rho4, OBSERVABLES["E_TFIM"])) < 1e-9
            p4 = born_tensor(rho4, povms["icosahedron"]["E"])
            for x, v_marg in [(x_canonical(s_ico), var_can),
                              (x_letter(s_ico, rbar), var_obs),
                              (x_letter(s_ico, r_true), var_orc)]:
                v_full = exact_second_moment(
                    p4, OBSERVABLES["E_TFIM"], lut_uniform(x)) - e0 ** 2
                assert abs(v_full - v_marg) < 1e-9
            print("    n=4: marginal route == full outcome-tensor route "
                  "(all three levels): OK")

        gain = 1.0 - var_orc / var_can
        out[f"nscale/{n}/var"] = np.array([var_can, var_obs, var_orc])
        out[f"nscale/{n}/floor"] = np.array([floor])
        out[f"nscale/{n}/state"] = np.array([e0, r_true[0]])
        print(f"    {n:>2d} {e0:>10.5f} {r_true[0]:>8.5f} {var_can:>10.4f} "
              f"{var_obs:>10.4f} {var_orc:>10.4f} {floor:>8.4f} {gain:>6.1%}")

    g4 = 1 - out["nscale/4/var"][2] / out["nscale/4/var"][0]
    g16 = 1 - out["nscale/16/var"][2] / out["nscale/16/var"][0]
    print(f"  frame condition / disjoint-pair dual-independence at every "
          "size: OK (asserted)\n  oracle gain converges: "
          f"{g4:.1%} (n=4) -> {g16:.1%} (n=16); floor share "
          f"{out['nscale/16/floor'][0] / out['nscale/16/var'][0]:.1%} at n=16")
    return out


nscale_out = nscaling(povms, states)


n-scaling: exact variance of the factorized TFIM-energy estimator, icosahedron,
n = 4..16 (sparse ground states; canonical / observable-opt / oracle):
     n         E0      <X>    Var can    Var obs    Var orc    floor    gain
    n=4: marginal route == full outcome-tensor route (all three levels): OK
     4   -5.22625  0.65328    49.6569    42.3000    41.9291   0.8284  15.6%


     6   -7.72741  0.64395    73.5949    63.0309    62.3535   1.1068  15.3%


     8  -10.25166  0.64073    97.7502    83.8575    82.9170   1.3990  15.2%
    10  -12.78491  0.63925   121.9776   104.7147   103.5243   1.7020  15.1%


    12  -15.32260  0.63844   146.2385   125.5864   124.1514   2.0111  15.1%


    14  -17.86281  0.63796   170.5177   146.4663   144.7894   2.3240  15.1%


    16  -20.40459  0.63764   194.8082   167.3516   165.4339   2.6393  15.1%
  frame condition / disjoint-pair dual-independence at every size: OK (asserted)
  oracle gain converges: 15.6% (n=4) -> 15.1% (n=16); floor share 1.4% at n=16


## 11. The receipts

The cells above *are* the module — lifted mechanically at build time — but a committed notebook
can drift after later module edits. This cell closes the gap behaviorally and numerically:

1. **behavioral spot-checks**: import the production module and assert our (identical-by-lift)
   primitives return bit-equal results on shared inputs;
2. **the full replay**: rebuild the exact dictionary `main()` writes — same loops, same key
   grammar — and compare against the committed `data/shadow_experiments.npz`: the key sets must
   match exactly, and every value must agree to **zero** (not tolerance — the notebook threaded
   the same seed through the same operations in the same order).

The notebook never writes the npz; regenerating it is the script's job
(`uv run shadow_experiments.py`, deterministic).

In [25]:
# === Anti-drift: this notebook == the committed npz, exactly ===

import shadow_experiments as se

# behavioral spot-checks across the pipeline
_p = povms["icosahedron"]
assert np.array_equal(born_tensor(states["TFIM"], _p["E"]),
                      se.born_tensor(states["TFIM"], _p["E"]))
_a, _b = optimize_dual(_p["s"], np.full(12, 1 / 12), 2)
_a2, _b2 = se.optimize_dual(_p["s"], np.full(12, 1 / 12), 2)
assert np.array_equal(_a, _a2) and np.array_equal(_b, _b2)
_R = np.load(DATA / "group_I.npz")["rotations"]
for f_nb, f_se in ((channel_R1, se.channel_R1), (channel_R2, se.channel_R2)):
    M_nb, m_nb = f_nb(_p["s"], _R, T_PROBE, t_PROBE)
    M_se, m_se = f_se(_p["s"], _R, se.T_PROBE, se.t_PROBE)
    assert np.array_equal(M_nb, M_se) and np.array_equal(m_nb, m_se)
assert np.array_equal(site_bloch(states["GHZ"]), se.site_bloch(states["GHZ"]))
print("behavioral spot-checks against the production module: OK")

# rebuild main()'s output dict -- same loops, same key grammar
out = {}
for (pn, sn, on), st in exp1.items():
    out[f"exp1/{pn}/{sn}/{on}"] = np.array(
        [st["bias2"], st["var"], st["mse"], st["se_mse"]])
for (pn, on), v in exp1_haar.items():
    out[f"exp1_haar/{pn}/{on}"] = np.array([v])
for (pn, sn, on, ln), st in exp2.items():
    out[f"exp2/{pn}/{sn}/{on}/{ln}"] = np.array(
        [st["bias2"], st["var"], st["mse"], st["se_mse"]])
for key, st in exp3.items():
    if key[1] == "eta":
        out[f"exp3/eta/{key[0]}"] = np.array(st)
    else:
        p_rate, on, ln = key
        out[f"exp3/{p_rate}/{on}/{ln}"] = np.array(
            [st["bias2"], st["var"], st["mse"], st["se_mse"]])
for (cn, p_rate), row in blind.items():
    out[f"blind/{cn}/{p_rate}"] = np.array(
        [row["eta"], *row["X0"], *row["Z0"]])
for label, v in aniso.items():
    out[f"blind/aniso/{label}"] = v
for extra in (exact_out, four_out, ratio_out, tp_out, gate_out, nscale_out):
    out.update(extra)

committed = np.load(DATA / "shadow_experiments.npz")
assert set(out) == set(committed.files), \
    sorted(set(out) ^ set(committed.files))[:10]
fams = {}
for k in committed.files:
    fams[k.split("/")[0]] = fams.get(k.split("/")[0], 0) + 1
print("families: " + ", ".join(f"{f} ({n})" for f, n in sorted(fams.items())))
worst = max(np.abs(out[k] - committed[k]).max() for k in committed.files)
print(f"\nnpz keys: {len(committed.files)}  (all present, none extra)")
print(f"value-for-value: max |notebook - committed| = {worst}")
assert worst == 0.0
print("\nThe committed npz and this notebook agree exactly. "
      "The notebook wrote nothing.")

behavioral spot-checks against the production module: OK
families: blind (12), exact (48), exact4 (5), exact_ratio (80), exp1 (90), exp1_haar (30), exp2 (120), exp3 (65), gatenoise (124), nscale (21), twirl1 (16), twirl2 (18)

npz keys: 629  (all present, none extra)
value-for-value: max |notebook - committed| = 0.0

The committed npz and this notebook agree exactly. The notebook wrote nothing.


## Closing notes

**What was shown.**

| § | finding |
|---|---|
| 3 | at the canonical dual the four antipodal solids are provably identical on every (state, observable) — the variance sees only vertex moments $\le 3$; single weight-$w$ strings sit at $3^w - \langle P\rangle^2$ exactly; the tetrahedron escapes in exactly one cell, and it takes three conditions at once |
| 4 | design strength pins the tails at the sphere's weight but does not minimize them: the cube ($\pm\sqrt3$ always) is lightest, the octahedron — standard Pauli-6 shadows — heaviest |
| 5 | dual optimization is a per-letter QP; gains are real, modest, and do not scale with $V$ (the cube beats both 5-designs); the per-letter oracle can lose exactly (1.069 — cross-term covariances the objective never sees); the oracle is a ceiling, not a protocol |
| 6 | $\mathbb E[\hat\eta] = (1-p)/3$ is an identity; calibration converts an incurable bias ($(1-p)^w$, exact) into a variance premium (~60% at $p=0$) that amortizes as $1/R_C$ |
| 7 | a $\ket 0$-calibration is structurally blind to channels fixing $\ket 0$; the reported bias is exactly $(Tr+t-r)_a$; $\hat v_\perp$ is a free tilt diagnostic; amplitude damping evades both; the failure mode is silence |
| 8 | one lemma, two conjugations: readout∘noise → $T_{zz}/3$ (randomized-projective), noise alone → $(\operatorname{tr}T/3)/3$ (twirled-native, offset dead twice over, SIC included); the estimator-channel factor identifies the protocol (0.62 vs 0.72 on one probe); $2T$ is the universal minimal twirl and the dodecahedron's full-$2I$ bill is a *realization* cost |
| 9 | gate noise breaks the twirl exactly as far as the drawn words: projective residuals are linear and track the draw–noise correlation (min-magic recompilation *raises* the bill); the $2T$ draw halves the icosahedron's exposure; the twirled-native $Z_0$ residual is second order — proven: prefix conjugation + signed permutations ⇒ diagonal first-order error, which one scalar absorbs along $z$ and the word set's displacement balance finishes — and its one row prices all five solids, the SIC's first noise numbers |
| 10 | everything holds at $n = 16$, exactly: gains converge (15.1%/14.1%), the dual-independent floor stays $\le 1.7\%$, and the ceiling is the factorized class itself |

**The study's assumptions, collected** (each is load-bearing somewhere above): the per-qubit
*factorized* estimator class (§3, §10 — the ceiling); i.i.d. measurement-side noise (§6);
the depolarizing family for the one-scalar calibration — *assumed* under the native
implementation, *enforced* under either randomized one (§7, §8); a trusted $\ket 0$ probe
(§6; relaxing it is Jeanette et al.'s blind calibration); the twirl's $g$-independence
(§9 prices its failure); clean classical post-processing throughout.

**The three corners** (the appendix's closing trade): *native* — no random gates, keeps the SIC,
but the depolarizing model is an assumption and the calibration has a structural blind spot;
*twirled-native* — one all-Clifford depth-$\le2$ word per shot on top of the dilation turns the
model into a theorem, keeps the SIC, and confines the correlated exposure to that word;
*randomized-projective* — sheds the dilation, but one draw must realize *and* twirl: no SIC, an
inexact alignment everywhere but the octahedron, and the dodecahedron's $0.8\,\Phi$ per shot.
Which corner wins is a property of the hardware; the mathematics fixes the menu.

**Numbers worth holding cold** (everything else: know *where it lives*, not its digits): the
ideal shrinkage $1/3$; *that* the two protocols read different scalars off the same noise (0.62
vs 0.72 as an existence proof); the ~15% factorized-optimization ceiling; the dodecahedron's
$0.8\,\Phi$ per shot; the $2T$ draw = twelve all-Clifford words of depth $\le 2$.

**Self-test.** This notebook is built to be read closed-book: take each claim above,
regenerate it on paper, and only then read the derivation back. One of them — *why is
the twirled-native $Z_0$ residual second order in $\gamma$?* — is the one §9 works out in full,
mechanism and verification.

To run the production script end to end (writes the npz; deterministic):

```
cd code && uv run shadow_experiments.py
```

To regenerate this notebook after editing the builder (never edit the .ipynb directly):

```
cd code && uv run python _build_shadow_walkthrough.py
uv run --with jupyter --with nbconvert jupyter nbconvert --to notebook --execute --inplace shadow_walkthrough.ipynb
```